# 17 - מוקדי מעבר רב-אופניים (Multimodal Transfer Hubs)

מחברת 14 ביססה כי קובץ ה-GTFS הישראלי אינו רשת אחת אלא שש, אחת לכל `route_type`, ומדדה כל אחת מהן בנפרד. מחברת זו שואלת את השאלה המשלימה: **היכן שש הרשתות הללו נוגעות זו בזו?**

תחנה המשורתת ביותר מאופן תחבורה (mode) אחד היא *מוקד מעבר* (transfer hub). זהו המקום היחיד שבו נוסע יכול לעבור בין אופני תחבורה ללא הליכה, ו - וזו הנקודה החשובה לפרויקט זה - זהו המקום היחיד שבו כשל פיזי בודד יכול להוציא משירות יותר מאופן תחבורה אחד בעת ובעונה אחת. סגירה של מפרץ אוטובוסים עולה לרשת האוטובוסים צומת אחד; סגירה של מסוף מעבר עולה לרשת האוטובוסים צומת **וגם** מנתקת את קו ההזנה הרכבתי התלוי בו.

לפיכך מחברת זו:

1. בונה מחדש, מתוך הקובץ הגולמי, את קבוצת אופני התחבורה המשרתים כל תחנה (`trips.txt` -> `routes.txt` -> `route_type`);
2. מדרגת תחנות לפי מספר אופני התחבורה שהן משרתות ולפי נפח השירות המתוזמן המשולב;
3. **בוחנת** האם תחנות רב-אופניות אכן חשובות מבנית יותר מתחנות חד-אופניות, באמצעות מבחן Mann-Whitney U על betweenness ומבחן Fisher מדויק על שיעור ה-articulation point, כאשר גדלי אפקט מדווחים לצד ערכי ה-p;
4. בונה **גרף תלות-הדדית בין אופני תחבורה** (mode-interdependence graph) שצמתיו הם אופני תחבורה ומשקלי קשתותיו הם מספר התחנות המשותפות;
5. ממפה גאוגרפית את המוקדים המובילים;
6. דנה במשמעות כל זאת לחוסן הרשת.

**שאלת המחקר.** אילו תחנות מחברות בין כמה אופני תחבורה, והאם הן קריטיות מבנית יותר לרשת המשולבת מאשר תחנות חד-אופניות רגילות?

**הממצא המרכזי, המוצג כבר בפתח כדי שניתן יהיה לקרוא את שאר המחברת אל מולו.** ברמת ה-`stop_id` של GTFS, אופני התחבורה בישראל כמעט אף פעם אינם חולקים תחנה. הרוב המכריע של התחנות משרת אופן תחבורה אחד בדיוק, וקומץ התחנות המשרתות שניים הן כמעט כולן אוטובוס בתוספת אוטובוס מבוסס-ביקוש - כלומר מציאות תחבורתית אחת העוטה שני קודים של GTFS. **רכבת ואוטובוס אינם חולקים אף ערך `stop_id` בקובץ זה.** אין זה משום שאין בישראל מעבר בין רכבת לאוטובוס; זהו משום שרציף רכבת ומפרץ האוטובוסים שמחוצה לו הם רשומות שונות עם מזהים שונים. לכן ההגדרה הקשיחה של `stop_id` משותף מודדת *מוסכמת רישום נתונים*, ולא *עובדה תחבורתית*, ועל כן המחברת מחשבת הגדרה שנייה, מרחבית - אופני תחבורה הנגישים במרחק הליכה קצר - ומריצה כל מבחן על שתיהן. שתי ההגדרות אינן מסכימות, ואי-ההסכמה הזו היא התוצאה המאלפת ביותר כאן.

## קלט

| נתיב | הופק על ידי | משמש ל |
|---|---|---|
| `outputs/nb/01_data_preparation/tables/routes_clean.csv` | מחברת 01 | `route_id -> route_type` |
| `outputs/nb/01_data_preparation/tables/trips_clean.csv` | מחברת 01 | `trip_id -> route_id` |
| `outputs/nb/01_data_preparation/tables/stops_clean.csv` | מחברת 01 | מקור קואורדינטות חלופי לתחנות שאינן צמתים בגרף |
| `outputs/nb/02_graph_construction/tables/nodes.csv` | מחברת 02 | `stop_name`, `lat`, `lon`, `region`, `stop_use_count` |
| `outputs/nb/03_descriptive_analysis/tables/articulation_points.csv` | מחברת 03 | אילו תחנות הן צמתי חיתוך |
| `outputs/nb/04_centrality_analysis/tables/stop_metrics.csv` | מחברת 04 | `approx_betweenness`, `degree` |
| `israel-public-transportation/stop_times.txt` | הקובץ הגולמי | באילו תחנות כל trip עוצר בפועל |

**מחברות שחייבות לרוץ קודם:** `01_data_preparation`, `02_graph_construction`, `03_descriptive_analysis`, `04_centrality_analysis`. כל אחת מאותרת לפי קידומת התיקייה הדו-ספרתית שלה, וכל מחברת חסרה מעלה `FileNotFoundError` הנוקב בשם המחברת שיש להריץ.

**תלות בנתונים חיצוניים.** `stop_times.txt` הוא בגודל 816 MB / כ-15.7M שורות ואינו מנוהל ב-git. תא ההורדה שלהלן מושך אותו מ-Google Drive בהרצה הראשונה; הקובץ נקרא בזרימה שורה אחר שורה ולעולם אינו נטען כטבלה.

## פלט (הכול תחת `outputs/nb/17_multimodal_transfer_hubs/`)

| נתיב | תוכן |
|---|---|
| `tables/transfer_hubs.csv` | **סכמה נדרשת**: `stop_id, stop_name, lat, lon, region, n_modes, modes_served, is_multimodal` - שורה אחת לכל תחנה בקובץ |
| `tables/mode_interdependence.csv` | **סכמה נדרשת**: `mode_a, mode_b, shared_stops` - כל זוג אופני תחבורה לא-מסודר |
| `multimodal_summary.json` | **נדרש**: מספרי המפתח, סטטיסטיקות המבחנים וגדלי האפקט |
| `tables/transfer_hub_details.csv` | אותן תחנות עם העמודות שאינן נכנסות לסכמה הנדרשת: נפח שירות, עצירות לפי אופן תחבורה, קבוצת אופני התחבורה במרחק הליכה, degree, betweenness, דגל articulation |
| `tables/mode_interdependence_walkable.csv` | הווריאנט המרחבי של טבלת התלות ההדדית (`mode_a, mode_b, interchange_stops`) |
| `tables/multimodal_structural_tests.csv` | כל מבחן השערה שהורץ כאן, עם גודל אפקט, רווח בר-סמך וערך p מתוקן Holm |
| `tables/stop_mode_calls.csv`, `stream_stats.json` | תוצאת הזרימה, המשמשת גם כמטמון של מחברת זו |
| `figures/hub_inventory.png`, `figures/mode_interdependence_graph.png`, `figures/transfer_hub_map.png`, `figures/structural_importance.png` | ארבעת האיורים |

דבר מחוץ לתיקייה זו אינו נכתב. התיקיות המצוטטות בדוח `outputs/tables`, `outputs/figures` ו-`outputs/rail` אינן נוגעות כלל.

## 1. אתחול סביבת העבודה

זהה לכל מחברת אחרת בסדרה, כך שכל הסדרה רצה באותו אופן מקומית וגם ב-Google Colab. `_ensure(...)` מתקין ב-pip רק חבילות שחסרות באמת, `find_repo_root()` מטפס כלפי מעלה מתיקיית העבודה בחיפוש אחר תיקיית ה-GTFS (ומשכפל את המאגר אם אנו ב-Colab), ולאחר מכן התא קובע את `REPO`, `DATA` ו-`OUT`. כל מה שלהלן תלוי בשלושת הנתיבים הללו, ולכן תא זה חייב לרוץ ראשון.

In [ ]:
# --- Environment bootstrap (safe to re-run, works locally and on Google Colab) ---
import os, sys, subprocess
from pathlib import Path

def _ensure(*pkgs):
    """Install only the packages that are actually missing."""
    import importlib.util
    alias = {"scikit-learn": "sklearn", "python-louvain": "community",
             "python-bidi": "bidi", "node2vec": "node2vec"}
    missing = [p for p in pkgs
               if importlib.util.find_spec(alias.get(p, p.replace("-", "_"))) is None]
    if missing:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", *missing], check=True)

def find_repo_root():
    """Find the repo locally; on Colab, clone it."""
    here = Path(os.getcwd()).resolve()
    for cand in [here, *here.parents]:
        if (cand / "israel-public-transportation").is_dir():
            return cand
    target = Path("/content/israel-transit-network-resilience")
    if not target.exists():
        subprocess.run(["git", "clone", "--depth", "1",
                        "https://github.com/seanfourman/israel-transit-network-resilience.git",
                        str(target)], check=True)
    return target

REPO = find_repo_root()
os.chdir(REPO)
DATA = REPO / "israel-public-transportation"
OUT = REPO / "outputs" / "nb"
OUT.mkdir(parents=True, exist_ok=True)
print("Repo root:", REPO)

## 2. ספריות, תיקיות השלב, בקרות עלות ואוצר המילים של אופני התחבורה

ערימת הכלים המדעית בתוספת `scipy.stats` (מבחני ההשערות), `scipy.spatial` (חיפוש השכנות במרחק הליכה) ומודול `csv` מהספרייה הסטנדרטית, שהוא זה שקורא בפועל את הקובץ בן 816 MB שורה אחר שורה. שלב זה מחזיק בדיוק בתיקייה אחת, `outputs/nb/17_multimodal_transfer_hubs/`, ובתוכה `tables/` ו-`figures/`.

**בקרות עלות.** הפעולה היקרה היחידה היא מעבר הזרימה על `stop_times.txt`: כ-**3-6 דקות**, שכן יש לפרסר את כל כ-15.7M השורות. התוצאה דטרמיניסטית, ולכן היא נשמרת במטמון אל `tables/stop_mode_calls.csv` ונטענת מחדש בכשנייה בהרצה חוזרת; `FORCE_RESTREAM = True` עוקף את המטמון. כל מה שאחרי כן נמשך שניות: עץ k-d מעל כ-30k נקודות וקומץ מבחני דירוג.

**`WALK_RADIUS_M = 150`** הוא הפרמטר השרירותי האמיתי היחיד במחברת, והוא הרדיוס המשמש להגדרה המרחבית של "אותו מסוף מעבר". 150 מ' הם בקירוב שתי דקות הליכה, וזה המרחק שבו רציף רכבת, הרחבה שלפניו ומפרצי האוטובוסים שבחוץ חדלים להיות מקומות נפרדים. סעיף 12 מריץ מחדש את כל הניתוח המרחבי ב-100 מ' וב-250 מ', כדי שהקורא יראה עד כמה המסקנה תלויה בבחירה זו.

**תוויות אופני תחבורה.** ששת קודי ה-`route_type` הקיימים בקובץ זה מפורטים במפורש, מועתקים מילה במילה ממחברת 14 כדי ששתי המחברות יסכימו, ומסויגים במתכוון היכן שקוד ה-GTFS מטעה: `8` הוא נומינלית "trolleybus" אך בקובץ זה משמש מפעילי מוניות שירות, ו-`715` הוא קוד ה-GTFS ל"שירות אוטובוס מבוסס ביקוש". כל קוד בלתי צפוי מתויג `other (<code>)` במקום להיות מושמט בשקט.

In [ ]:
# --- Libraries, stage folders and cost knobs ------------------------------
_ensure('pandas', 'numpy', 'networkx', 'matplotlib', 'seaborn', 'scipy')

import csv, json, time
from collections import Counter, defaultdict
from itertools import combinations

import numpy as np
import pandas as pd
import networkx as nx
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.spatial import cKDTree
from scipy.stats import mannwhitneyu, chi2_contingency, fisher_exact

sns.set_theme(style='whitegrid', font_scale=1.05)

# A handful of rows in stop_times.txt are very long; raise the csv field limit up front.
csv.field_size_limit(10_000_000)

# --- Stage output folders -------------------------------------------------
STAGE = OUT / '17_multimodal_transfer_hubs'
TABLES = STAGE / 'tables'
FIGURES = STAGE / 'figures'
TABLES.mkdir(parents=True, exist_ok=True)
FIGURES.mkdir(parents=True, exist_ok=True)

# --- Cost knobs and analysis parameters ----------------------------------
FORCE_RESTREAM = False        # True -> always re-read the 816 MB feed (3-6 minutes)
PROGRESS_EVERY = 2_000_000    # progress print interval during the streaming pass
FIG_DPI = 150                 # figure resolution; drop to 90 for smaller files
WALK_RADIUS_M = 150           # "same interchange" radius for the spatial definition
WALK_RADIUS_SENSITIVITY = (100, 150, 250)   # radii re-tested in section 12
TOP_N_HUBS = 15               # rows in the printed / plotted hub rankings
MAP_ANNOTATE = 10             # hub labels drawn on the map
SEED = 42

# --- GTFS mode vocabulary (labels match notebooks 11 and 14) --------------
MODE_LABELS = {
    '0': 'tram/light rail',
    '2': 'rail',
    '3': 'bus',
    '5': 'cable tram',
    '8': 'trolleybus/taxi-coded',
    '715': 'demand/other bus',
}
UNKNOWN_MODE = 'unknown'

def mode_label(route_type):
    """Human-readable name for a GTFS route_type code; never raises, never drops."""
    code = str(route_type).strip()
    if code == '' or code.lower() == 'nan':
        return UNKNOWN_MODE
    return MODE_LABELS.get(code, f'other ({code})')

print('stage folder  :', STAGE)
print('modes tracked :', ', '.join(MODE_LABELS.values()))
print('walk radius   :', WALK_RADIUS_M, 'm')

## 3. רינדור תוויות בעברית

שמות התחנות בקובץ הישראלי הם בעברית, ודירוגי המוקדים והמפה שלהלן מתויגים בהם. Matplotlib אינה מממשת את אלגוריתם הדו-כיווניות של Unicode, ולכן טקסט מימין לשמאל מצויר הפוך. התא מתקן פעם אחת את `matplotlib.text.Text.set_text` כך שכל מחרוזת המכילה עברית מומרת לסדר תצוגה באמצעות `python-bidi`, ובוחר גופן בעל גליפים עבריים. התיקון אידמפוטנטי, ולכן הרצה חוזרת אינה מערימה תיקונים זה על זה. מכיוון שהתיקון גלובלי, יש להעביר ל-matplotlib מכאן ואילך מחרוזות עבריות גולמיות - קריאה נוספת ל-`fix_he()` באופן ידני הייתה הופכת את הטקסט פעמיים.

In [ ]:
# Stop names are Hebrew. Matplotlib does not apply the Unicode bidi algorithm, so
# Hebrew labels render reversed. Patch it once, before drawing any figure.
_ensure("python-bidi")
import re
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.text as mtext
from bidi.algorithm import get_display

_HEBREW_RE = re.compile(r"[\u0590-\u05FF]")

def fix_he(text):
    """Return display-ordered text. Non-Hebrew is returned untouched."""
    if not isinstance(text, str) or not _HEBREW_RE.search(text):
        return text
    return get_display(text)

def install_hebrew():
    # Arial exists on Windows; DejaVu Sans ships with matplotlib and covers Hebrew.
    matplotlib.rcParams["font.family"] = ["Arial", "DejaVu Sans"]
    matplotlib.rcParams["axes.unicode_minus"] = False
    if getattr(mtext.Text, "_bidi_patched", False):
        return
    _orig = mtext.Text.set_text
    def set_text(self, s):
        if isinstance(s, str) and getattr(self, "_bidi_display", None) == s:
            return _orig(self, s)
        fixed = fix_he(s)
        if isinstance(fixed, str):
            self._bidi_display = fixed
        return _orig(self, fixed)
    mtext.Text.set_text = set_text
    mtext.Text._bidi_patched = True

install_hebrew()

## 4. איתור השלבים הקודמים

ארבעה שלבים קודמים מזינים שלב זה. הם מאותרים לפי **הקידומת הדו-ספרתית** שלהם ולא לפי slug מדויק, כך שתיקייה ששמה שונה מ-`04_centrality_analysis` לכל שם אחר המתחיל ב-`04` עדיין תימצא. אם תיקייה או קובץ חסרים, פונקציית העזר מעלה `FileNotFoundError` הנוקב בשם המחברת שיש להריץ - נסיגה שקטה כאן הייתה מייצרת טבלת מוקדים שבה כל תחנה מתויגת `unknown`, או מבחן מבני המשווה בין שתי קבוצות ריקות, ושניהם נראים סבירים ושניהם שגויים לחלוטין.

מזהי GTFS נקראים בכל מקום כ**מחרוזות**. הנחת טיפוסים אוטומטית של pandas הייתה הופכת את `route_type` למספר שלם, מסירה אפסים מובילים ממזהי תחנות, ושוברת כל join שלהלן. טבלת המרכזיות היא היוצא מן הכלל היחיד: עמודות המדדים שלה חייבות להישאר מספריות, ולכן רק `stop_id` נכפה שם למחרוזת.

שימו לב למלכודת שם העמודה בשלב 04: עמודת ה-betweenness היא `approx_betweenness` ולא `betweenness` - שלב 04 מחשב אותה בקירוב מבוסס דגימת מקורות, והשם מתעד זאת. התא בודק את קיומה במפורש במקום להיכשל מאוחר יותר עם `KeyError` באמצע מבחן.

In [ ]:
# --- Resolve earlier stages by their two-digit prefix ---------------------
def find_stage(prefix, notebook_hint):
    """Return an earlier stage's output folder, matched by its NN prefix."""
    matches = sorted(p for p in OUT.glob(f'{prefix}*') if p.is_dir())
    if not matches:
        raise FileNotFoundError(
            f'No stage folder starting with "{prefix}" under {OUT}. '
            f'Run notebook {notebook_hint} first.')
    return matches[0]


def stage_artifact(prefix, filename, notebook_hint):
    """Path of `filename` inside stage `prefix`, or a FileNotFoundError that says why."""
    stage = find_stage(prefix, notebook_hint)
    direct = stage / 'tables' / filename
    if direct.exists():
        return direct
    hits = sorted(stage.rglob(filename))
    if not hits:
        raise FileNotFoundError(
            f'{filename} not found under {stage}. '
            f'Run notebook {notebook_hint} first - it writes {filename}.')
    return hits[0]


def read_gtfs_table(path):
    """GTFS ids are opaque codes: read every column as a string, keep blanks as ''."""
    return pd.read_csv(path, dtype=str, keep_default_na=False, encoding='utf-8-sig')


def read_metric_table(path):
    """Keep stop_id a string but let the metric columns stay numeric."""
    return pd.read_csv(path, dtype={'stop_id': str}, encoding='utf-8-sig')


routes = read_gtfs_table(stage_artifact('01', 'routes_clean.csv', '01_data_preparation'))
trips = read_gtfs_table(stage_artifact('01', 'trips_clean.csv', '01_data_preparation'))
stops = read_gtfs_table(stage_artifact('01', 'stops_clean.csv', '01_data_preparation'))
nodes = read_metric_table(stage_artifact('02', 'nodes.csv', '02_graph_construction'))
artic = read_metric_table(stage_artifact('03', 'articulation_points.csv',
                                         '03_descriptive_analysis'))
metrics = read_metric_table(stage_artifact('04', 'stop_metrics.csv',
                                           '04_centrality_analysis'))

# Column contracts, checked here so a rename upstream fails loudly and immediately.
for name, frame, needed in [
        ('routes_clean.csv', routes, ['route_id', 'route_type']),
        ('trips_clean.csv', trips, ['trip_id', 'route_id']),
        ('nodes.csv', nodes, ['stop_id', 'stop_name', 'lat', 'lon', 'region']),
        ('articulation_points.csv', artic, ['stop_id']),
        ('stop_metrics.csv', metrics, ['stop_id', 'degree', 'approx_betweenness'])]:
    missing = [c for c in needed if c not in frame.columns]
    if missing:
        raise KeyError(f'{name} is missing column(s) {missing} - re-run the notebook '
                       f'that produces it.')

BETWEENNESS = 'approx_betweenness'   # stage 04 names it this, NOT "betweenness"
articulation_ids = set(artic['stop_id'])

print(f'routes {len(routes):,} | trips {len(trips):,} | stops {len(stops):,}')
print(f'graph nodes {len(nodes):,} | centrality rows {len(metrics):,} | '
      f'articulation points {len(articulation_ids):,}')
print('route_type codes present:', sorted(set(routes['route_type'])))

## 5. מיפוי `trip_id -> mode`

אופן התחבורה נמצא ב-`routes.txt`, אך הקובץ שעלינו לקרוא בזרימה (`stop_times.txt`) מכיר רק `trip_id`. הגשר הוא `trips.txt`, ולכן - בדיוק כמו במחברת 14 - שני החיפושים מורכבים פעם אחת, בזיכרון, למילון יחיד `trip_id -> תווית אופן תחבורה` (כ-420k רשומות, זניח לעומת 816 MB שאנו עומדים לקרוא). כל עצירת תחנה שנמצאת במהלך מעבר הזרימה מנותבת אז לאופן התחבורה שלה בחיפוש מילוני יחיד.

אותו join מספק גם את נפחי הקווים והנסיעות לכל אופן תחבורה. אלה מודפסים אל מול הערכים שאומתו במחברת 14, כך שחיבור שנשבר בשקט ייתפס כאן ולא שלושה תאים מאוחר יותר: אוטובוס `3` = 6,796 קווים / 412,544 נסיעות, רכבת `2` = 962 / 1,188, מבוסס-ביקוש `715` = 14 / 458, רכבת קלה `0` = 8 / 2,890, trolleybus `8` = 8 / 47, רכבל `5` = 4 / 3,006. "רשומת route" ב-GTFS היא כיוון-ווריאנט של קו, ולא קו - ומכאן 962 "קווי" רכבת עבור רשת של כ-67 תחנות.

נסיעות שה-`route_id` שלהן נעדר מ-`routes_clean.csv` היו הופכות בשקט ל-`unknown`, ולכן הן נספרות במפורש במקום להיזרק.

In [ ]:
# --- route_id -> route_type -> mode label, then trip_id -> mode label -----
route_type_of = dict(zip(routes['route_id'], routes['route_type']))
trip_mode = {trip: mode_label(route_type_of.get(route, ''))
             for trip, route in zip(trips['trip_id'], trips['route_id'])}
orphan_trips = sum(1 for route in trips['route_id'] if route not in route_type_of)

trips_typed = trips.assign(route_type=trips['route_id'].map(route_type_of).fillna(''))
mode_meta = pd.DataFrame({
    'routes': routes.groupby('route_type').size(),
    'trips': trips_typed.groupby('route_type').size(),
}).fillna(0).astype(int)
mode_meta.index.name = 'route_type'
mode_meta = mode_meta.reset_index()
mode_meta['mode_label'] = mode_meta['route_type'].map(mode_label)
mode_meta = mode_meta.sort_values('trips', ascending=False).reset_index(drop=True)

# Values verified in notebook 14; printed side by side so a broken join is obvious.
EXPECTED = {'3': (6796, 412544), '2': (962, 1188), '715': (14, 458),
            '0': (8, 2890), '8': (8, 47), '5': (4, 3006)}
print(f'trip_id -> mode entries: {len(trip_mode):,}')
print(f'trips whose route_id is missing from routes_clean.csv: {orphan_trips:,}')
print()
print('{:<6}{:<24}{:>9}{:>10}{:>12}'.format('code', 'mode', 'routes', 'trips', 'matches 14'))
for _, row in mode_meta.iterrows():
    exp = EXPECTED.get(row['route_type'])
    ok = 'n/a' if exp is None else ('yes' if (row['routes'], row['trips']) == exp else 'NO')
    print('{:<6}{:<24}{:>9,}{:>10,}{:>12}'.format(
        row['route_type'], row['mode_label'], row['routes'], row['trips'], ok))

## 6. תלות בנתונים חיצוניים: `stop_times.txt`

`stop_times.txt` הוא בגודל 816 MB - הרבה מעבר למגבלת גודל הקובץ של GitHub - ולכן הוא **אינו** במאגר. התא שלהלן מוריד אותו מ-Google Drive בהרצה הראשונה ומדלג על ההורדה אם הקובץ כבר קיים. זוהי תלות הרשת החיצונית היחידה של המחברת; כל השאר נמצא במאגר או מיוצר על ידי מחברת קודמת. ההורדה נמשכת כמה דקות בהרצה ראשונה ב-Colab.

In [ ]:
# stop_times.txt is 816MB and is not tracked in git - fetch it on demand.
_ensure("gdown")
import gdown
STOP_TIMES = DATA / "stop_times.txt"
if not STOP_TIMES.exists():
    gdown.download(id="1V_yPAWXV6mGTFGrfiosah5LngcLZnviW",
                   output=str(STOP_TIMES), quiet=False)
print("stop_times.txt:", round(STOP_TIMES.stat().st_size / 1024**2, 1), "MB")

## 7. מעבר זרימה אחד: עצירות מתוזמנות לכל (אופן תחבורה, תחנה)

זהו התא היקר: **3-6 דקות** עבור כ-15.7M שורות. הקובץ נקרא **שורה אחר שורה** באמצעות `csv.reader` ולעולם אינו ממומש כטבלה - בכ-15.7M שורות, טעינה של הקובץ כולו ב-pandas הייתה דורשת כמה ג'יגה-בייט עבור תוצאה שניתן לצבור בכמה מגה-בייט של מונים.

העבודה לכל שורה היא: לאתר את אופן התחבורה של הנסיעה (פעם אחת לכל בלוק נסיעה, לא פעם אחת לכל שורה, מכיוון שהקובץ ממוין לפי `(trip_id, stop_sequence)` - אומת על הקובץ המלא במחברת 02), ואז להגדיל את `stop_calls[mode][stop_id]`. המצב הנשמר בסך הכול הוא O(modes x stops) ~ 30k רשומות, ולא O(rows). **עצירה מתוזמנת** (scheduled stop call) היא שורה אחת של `stop_times.txt`: נסיעה אחת העוצרת פעם אחת בתחנה אחת. זוהי יחידת נפח השירות הטבעית כאן, וזו אותה כמות ששלב 02 מכנה `stop_use_count`, רק מפוצלת לפי אופן תחבורה.

**במחברת זו לא מתבצע שום פרסור של זמנים.** אנו זקוקים רק ל*אילו* תחנות נסיעה עוצרת, לעולם לא *מתי*, ולכן מלכודת ה"שעות >= 24" של GTFS אינה מתעוררת כלל. (`25:30:00` הוא זמן GTFS חוקי שמשמעותו 01:30 ביום השירות הבא; `datetime.strptime` דוחה אותו על הסף. כל מקום בפרויקט זה שכן זקוק לשעות שעון מפרסר אותן אריתמטית כ-`int(h) * 3600 + int(m) * 60 + int(s)`, מה שמטפל נכון בשעה 25 - וזו בעייתן של מחברות 18 ו-19, לא שלנו.)

נסיעות הנעדרות מ-`trips_clean.csv` נספרות תחת `unknown` במקום להיזרק, כך שאף עצירה אינה אובדת. התוצאה דטרמיניסטית ונשמרת במטמון אל `tables/stop_mode_calls.csv`; `FORCE_RESTREAM = True` כופה קריאה מחדש.

In [ ]:
# --- One pass over stop_times.txt, counting stop calls per (mode, stop) ---
STOP_MODE_CALLS = TABLES / 'stop_mode_calls.csv'
STREAM_STATS = STAGE / 'stream_stats.json'


def stream_stop_mode_calls(path, trip_mode, progress_every=PROGRESS_EVERY):
    """Scheduled stop calls per (mode, stop), in a single row-by-row pass.

    No time parsing is performed, so GTFS hours >= 24 are irrelevant here.
    Memory is O(modes x stops), never O(rows).
    """
    stop_calls = defaultdict(Counter)   # mode -> Counter[stop_id] = scheduled stop calls
    trips_observed = Counter()          # mode -> distinct trips met in the feed
    rows_by_mode = Counter()
    rows_read = 0
    t0 = time.time()

    with open(path, encoding='utf-8-sig', newline='') as handle:
        reader = csv.reader(handle)
        header = next(reader)
        for field in ('trip_id', 'stop_id'):
            if field not in header:
                raise ValueError(f'stop_times.txt has no "{field}" column')
        i_trip, i_stop = header.index('trip_id'), header.index('stop_id')

        prev_trip, mode = None, UNKNOWN_MODE
        for row in reader:
            rows_read += 1
            trip, stop = row[i_trip], row[i_stop]
            if trip != prev_trip:
                # New trip block: resolve the mode once per trip, not once per row.
                mode = trip_mode.get(trip, UNKNOWN_MODE)
                trips_observed[mode] += 1
                prev_trip = trip
            rows_by_mode[mode] += 1
            stop_calls[mode][stop] += 1
            if progress_every and rows_read % progress_every == 0:
                print(f'    {rows_read:,} rows | {time.time() - t0:,.0f}s')

    stats = {
        'rows_read': rows_read,
        'rows_by_mode': dict(rows_by_mode),
        'trips_observed': dict(trips_observed),
        'elapsed_seconds': round(time.time() - t0, 1),
    }
    return stop_calls, stats


cache_ready = STOP_MODE_CALLS.exists() and STREAM_STATS.exists()
if cache_ready and not FORCE_RESTREAM:
    print('Re-using the cached (mode, stop) counters from an earlier run of this notebook.')
    _cached = pd.read_csv(STOP_MODE_CALLS, dtype=str, keep_default_na=False,
                          encoding='utf-8-sig')
    stop_calls = defaultdict(Counter)
    for m, s, c in zip(_cached['mode_label'], _cached['stop_id'], _cached['stop_calls']):
        stop_calls[m][s] = int(c)
    with open(STREAM_STATS, encoding='utf-8') as handle:
        stream_stats = json.load(handle)
else:
    print('Streaming stop_times.txt (~15.7M rows) - this takes a few minutes ...')
    stop_calls, stream_stats = stream_stop_mode_calls(STOP_TIMES, trip_mode)

print('rows read       : {:,}'.format(stream_stats['rows_read']))
print('elapsed seconds :', stream_stats['elapsed_seconds'])
print()
print('{:<24}{:>16}{:>10}{:>9}'.format('mode', 'stop-time rows', 'trips', 'stops'))
for m, n_rows in sorted(stream_stats['rows_by_mode'].items(), key=lambda kv: -kv[1]):
    print('{:<24}{:>16,}{:>10,}{:>9,}'.format(
        m, n_rows, stream_stats['trips_observed'].get(m, 0), len(stop_calls.get(m, {}))))

## 8. שמירת תוצאת הזרימה

הטבלה בפורמט הארוך `(mode, stop, stop_calls)` נכתבת מיד, לפני כל ניתוח. היא משרתת שתי מטרות: היא המטמון שהופך הרצה חוזרת של מחברת זו לזולה, והיא מסלול הביקורת עבור כל מספר שלהלן - אם דירוג מוקד נראה שגוי, קובץ זה אומר בדיוק כמה עצירות של איזה אופן תחבורה ייחס מעבר הזרימה לאותה תחנה. `stream_stats.json` נושא את מוני השורות והנסיעות בצורה קריאה למכונה.

כל קובצי ה-CSV במחברת זו נכתבים כ-UTF-8 עם BOM, כך ששמות תחנות בעברית ייפתחו כראוי ב-Excel.

In [ ]:
# --- Persist the streaming result (also serves as the cache) --------------
stop_mode_calls = pd.DataFrame(
    [{'mode_label': m, 'stop_id': s, 'stop_calls': int(c)}
     for m, counter in stop_calls.items() for s, c in counter.items()]
)
stop_mode_calls.to_csv(STOP_MODE_CALLS, index=False, encoding='utf-8-sig')
with open(STREAM_STATS, 'w', encoding='utf-8') as handle:
    json.dump(stream_stats, handle, ensure_ascii=False, indent=2)

print(f'{len(stop_mode_calls):,} (mode, stop) rows -> {STOP_MODE_CALLS}')
print(f'streaming counters                -> {STREAM_STATS}')
print(f'distinct stops seen in the feed   : {stop_mode_calls["stop_id"].nunique():,}')
stop_mode_calls.head()

## 9. אילו אופני תחבורה משרתים כל תחנה

הטבלה הארוכה מקובצת לשורה אחת לכל תחנה: כמה אופני תחבורה נבדלים משרתים אותה (`n_modes`), אילו מהם (`modes_served`, רשימה ממוינת אלפביתית ומופרדת ב-`|` כך שהערך יציב ונוח להשוואת גרסאות), ונפח השירות המשולב (`total_stop_calls`, מסוכם על פני אופני התחבורה). `is_multimodal` הוא פשוט `n_modes >= 2`.

תכונות התחנה מגיעות מ-`nodes.csv` של שלב 02, שהיא הטבלה הסמכותית ל-`stop_name`, `lat`, `lon` ו-`region` בפרויקט זה. מספר תחנות מופיעות ב-`stop_times.txt` אך לא ב-`nodes.csv` - שלב 02 הופך תחנה לצומת בגרף רק אם קיימת נסיעה שאכן נעה אליה או ממנה, ולכן תחנה שהיא העצירה היחידה של נסיעה מנוונת נותרת ללא קשת וללא צומת. תחנות אלה **נשמרות** כאן (הן תחנות אמיתיות עם שירות אמיתי) ותכונותיהן מושלמות מ-`stops_clean.csv`; הספירה מודפסת ולא מוסתרת. מחוז נותר ריק כאשר אינו ידוע, במקום להיות מנוחש.

In [ ]:
# --- One row per stop: which modes serve it, and how much service ---------
per_stop = (stop_mode_calls
            .groupby('stop_id')
            .agg(n_modes=('mode_label', 'nunique'),
                 modes_served=('mode_label', lambda s: '|'.join(sorted(set(s)))),
                 total_stop_calls=('stop_calls', 'sum'))
            .reset_index())

# Primary attributes: stage 02 nodes.csv. Fallback: stage 01 stops_clean.csv.
node_attrs = nodes[['stop_id', 'stop_name', 'lat', 'lon', 'region']].copy()

fallback = stops[['stop_id', 'stop_name', 'stop_lat', 'stop_lon']].copy()
fallback['lat'] = pd.to_numeric(fallback['stop_lat'], errors='coerce')
fallback['lon'] = pd.to_numeric(fallback['stop_lon'], errors='coerce')
fallback['region'] = stops['region'] if 'region' in stops.columns else ''
fallback = fallback[['stop_id', 'stop_name', 'lat', 'lon', 'region']].drop_duplicates('stop_id')

hubs = per_stop.merge(node_attrs, on='stop_id', how='left')
missing_attrs = hubs['stop_name'].isna()
n_missing = int(missing_attrs.sum())
if n_missing:
    patch = fallback.set_index('stop_id')
    for column in ('stop_name', 'lat', 'lon', 'region'):
        hubs.loc[missing_attrs, column] = (hubs.loc[missing_attrs, 'stop_id']
                                           .map(patch[column]))
hubs['region'] = hubs['region'].fillna('')
hubs['stop_name'] = hubs['stop_name'].fillna('')
hubs['is_multimodal'] = hubs['n_modes'] >= 2
hubs = hubs.sort_values(['n_modes', 'total_stop_calls'], ascending=False).reset_index(drop=True)

print(f'stops in the feed                       : {len(hubs):,}')
print(f'  not vertices in stage 02 nodes.csv    : {n_missing:,} (attributes back-filled)')
print(f'  still without coordinates             : {int(hubs["lat"].isna().sum()):,}')
print(f'  multimodal (n_modes >= 2)             : {int(hubs["is_multimodal"].sum()):,}'
      f'  ({hubs["is_multimodal"].mean() * 100:.2f}% of stops)')
print()
print('stops by number of modes served:')
print(hubs['n_modes'].value_counts().sort_index().to_string())
print()
print('mode combinations actually observed at a single stop_id:')
print(hubs.loc[hubs['is_multimodal'], 'modes_served'].value_counts().to_string())

## 10. `tables/transfer_hubs.csv` (סכמה נדרשת) וטבלת הפרטים הנלווית

הפלט הנדרש הראשון, נכתב עם שמונה העמודות החוזיות בדיוק ולא יותר: `stop_id, stop_name, lat, lon, region, n_modes, modes_served, is_multimodal`. הוא נושא **שורה אחת לכל תחנה בקובץ**, ולא רק לרב-אופניות - `is_multimodal` הוא הדגל הבורר אותן, ומחברות המשך זקוקות למכנה לא פחות מאשר למונה.

כל מה שנמדד כאן מעבר לכך נכנס אל `tables/transfer_hub_details.csv` כדי שהחוזה יישאר נקי: נפח שירות משולב ולכל אופן תחבורה, קבוצת אופני התחבורה במרחק הליכה המחושבת בסעיף 12, והעמודות המבניות (`degree`, `approx_betweenness`, `is_articulation_point`) המצורפות משלבים 03 ו-04. טבלה נלווית זו נכתבת מאוחר יותר, לאחר שעמודות אלה קיימות.

In [ ]:
# --- tables/transfer_hubs.csv (exact required schema) --------------------
REQUIRED_HUB_COLUMNS = ['stop_id', 'stop_name', 'lat', 'lon', 'region',
                        'n_modes', 'modes_served', 'is_multimodal']
transfer_hubs = hubs[REQUIRED_HUB_COLUMNS].copy()
transfer_hubs.to_csv(TABLES / 'transfer_hubs.csv', index=False, encoding='utf-8-sig')

assert list(transfer_hubs.columns) == REQUIRED_HUB_COLUMNS, 'schema drift in transfer_hubs.csv'
print(f'{len(transfer_hubs):,} rows -> {TABLES / "transfer_hubs.csv"}')
transfer_hubs.head(8)

## 11. דירוג המוקדים: לפי מספר אופני התחבורה, ולפי נפח השירות

שני דירוגים, מכיוון שהם עונים על שאלות שונות - וכפי שההדפסה מראה, הם אינם מסכימים.

* **לפי מספר אופני התחבורה** הוא הדירוג המבני: כמה רשתות נפרדות נפגשות כאן. בקובץ זה מדובר בסדר כמעט מנוון, מכיוון ש-`n_modes` מקבל רק את הערכים 1 ו-2.
* **לפי נפח השירות המשולב** הוא הדירוג התפעולי: כמה שירות מתוזמן עובר דרך התחנה, מסוכם על פני אופני התחבורה הנוכחים. תחנה דו-אופנית עם 40 עצירות ביום היא נקודת מעבר בשם בלבד; תחנה דו-אופנית עם כמה אלפים היא מקום שבו סגירה מורגשת.

הפילוח לפי אופן תחבורה מודפס לצד זאת, כדי שהקורא יראה *מה* נפגש עם מה. כאן הממצא שהוכרז במבוא הופך למוחשי: זוג אופני התחבורה כמעט בכל `stop_id` רב-אופני הוא אוטובוס + אוטובוס מבוסס-ביקוש, כלומר תחנת אוטובוס שרשום עליה גם שירות לפי דרישה, ולא מסוף מעבר בין שתי מערכות תחבורה.

In [ ]:
# --- Two rankings of the multimodal stops --------------------------------
multimodal = hubs[hubs['is_multimodal']].copy()

calls_wide = (stop_mode_calls[stop_mode_calls['stop_id'].isin(set(multimodal['stop_id']))]
              .pivot_table(index='stop_id', columns='mode_label', values='stop_calls',
                           aggfunc='sum', fill_value=0)
              .reset_index())
ranked = (multimodal[['stop_id', 'stop_name', 'region', 'n_modes', 'modes_served',
                      'total_stop_calls']]
          .merge(calls_wide, on='stop_id', how='left')
          .sort_values(['n_modes', 'total_stop_calls'], ascending=False)
          .reset_index(drop=True))

print(f'multimodal stops: {len(multimodal):,}   max modes at one stop_id: '
      f'{int(hubs["n_modes"].max())}')
print(f'combined stop calls at multimodal stops: '
      f'{int(multimodal["total_stop_calls"].sum()):,} '
      f'({multimodal["total_stop_calls"].sum() / hubs["total_stop_calls"].sum() * 100:.2f}%'
      f' of all scheduled stop calls)')
print()
print(f'--- top {TOP_N_HUBS} multimodal stops by combined service volume ---')
print(ranked.head(TOP_N_HUBS).to_string(index=False))
print()
print('--- service volume of multimodal vs single-mode stops ---')
print(hubs.groupby('is_multimodal')['total_stop_calls']
      .agg(stops='size', median='median', mean='mean', total='sum').to_string())

## 12. גרף התלות ההדדית בין אופני התחבורה

הפלט הנדרש השני. הצמתים הם אופני תחבורה; משקל הקשת בין שני אופני תחבורה הוא מספר התחנות המשורתות על ידי **שניהם**. פורמלית, עבור אופני תחבורה $a$ ו-$b$ עם קבוצות תחנות $S_a$ ו-$S_b$, משקל הקשת הוא $|S_a \cap S_b|$ - הגרף הוא מבנה החיתוכים של שש קבוצות התחנות.

כל זוג לא-מסודר של אופני תחבורה שנצפו נכתב אל `tables/mode_interdependence.csv`, כולל הזוגות שמשקלם אפס. שורת אפס אינה מילוי סרק: "רכבת ואוטובוס חולקים בדיוק 0 מזהי תחנה" הוא המספר החשוב ביותר במחברת זו, וטבלה שהייתה משמיטה אותו הייתה מאפשרת לקורא להניח שהזוג פשוט לא נבדק.

בקריאה כגרף, הגרסה הקשיחה כמעט ריקה - קשת אחת, בין אוטובוס לאוטובוס מבוסס-ביקוש. זוהי התשובה הקשיחה הכנה, וסעיף 13 מסביר מדוע מדובר באמירה על אופן הרישום ב-GTFS ולא על התחבורה בישראל.

In [ ]:
# --- tables/mode_interdependence.csv (exact required schema) -------------
mode_stop_sets = {m: set(counter) for m, counter in stop_calls.items()}
observed_modes = sorted(mode_stop_sets, key=lambda m: -len(mode_stop_sets[m]))

interdependence = pd.DataFrame(
    [{'mode_a': a, 'mode_b': b,
      'shared_stops': len(mode_stop_sets[a] & mode_stop_sets[b])}
     for a, b in combinations(observed_modes, 2)]
).sort_values('shared_stops', ascending=False).reset_index(drop=True)
interdependence.to_csv(TABLES / 'mode_interdependence.csv', index=False,
                       encoding='utf-8-sig')

print('stops served, by mode:')
for m in observed_modes:
    print(f'  {m:<24}{len(mode_stop_sets[m]):>8,}')
print()
print(f'{len(interdependence)} mode pairs -> {TABLES / "mode_interdependence.csv"}')
print(interdependence.to_string(index=False))
print()
print('pairs sharing at least one stop_id: '
      f'{int((interdependence["shared_stops"] > 0).sum())} of {len(interdependence)}')

## 13. הגדרה שנייה: אופני תחבורה במרחק הליכה

שיתוף `stop_id` הוא מבחן קשיח, ובקובץ זה הוא המבחן הלא נכון. רציף רכבת, תחנת הרכבת הקלה ברחוב שבחוץ ומפרצי האוטובוסים ברחבה הם שלוש רשומות GTFS שונות בשלוש קואורדינטות שונות במקצת. שום זהירות בחיבור `trips -> routes` לא תגרום להן לחלוק מזהה, מכיוון שהמפעיל מעולם לא התכוון לכך.

לכן אנו מוסיפים הגדרה מרחבית. עבור כל תחנה, ניקח את איחוד אופני התחבורה המשורתים על ידי כל תחנה שבמרחק `WALK_RADIUS_M` מטרים - כולל היא עצמה - ונכנה זאת **קבוצת אופני התחבורה במרחק הליכה** (walkable mode set). תחנה שקבוצת ההליכה שלה כוללת שני אופני תחבורה או יותר היא *תחנת מעבר* (interchange stop): נוסע העומד בה יכול להגיע ברגל לאופן תחבורה אחר תוך שתי דקות.

שלוש הערות מימוש:

* **היטל.** הקואורדינטות מומרות למטרים באמצעות קירוב equirectangular מקומי (`x = lon * cos(lat0) * 111,320`, `y = lat * 110,540`) סביב קו הרוחב הממוצע של הקובץ. על פני מדינה בגובה 400 ק"מ, שגיאת המרחק המושרית היא הרבה מתחת לאחוז - חסרת משמעות בסף של 150 מ', וזולה בהרבה מחישוב גאודזי מלא עבור כ-30k נקודות.
* **שכנות, ולא clustering.** כל תחנה מקבלת רדיוס משלה. החלופה - מיזוג טרנזיטיבי של כל זוג תחנות במרחק 150 מ' ל"מתחם תחנות" - משתרשרת לאורך רצועות ארוכות של תחנות אוטובוס צפופות ומייצרת מתחמים באורך מאות מטרים, מה שהיה מנפח את הקבוצה הרב-אופנית בתחנות שאינן קרובות כלל למסוף מעבר. שכנות ברמת התחנה הבודדת אינה יכולה להשתרשר.
* **הרדיוס שרירותי, ולכן השפעתו נמדדת.** התא מריץ את הספירה מחדש ב-100, 150 ו-250 מ' ומדפיס את שלושתן. אילו המסקנה הייתה מתקיימת רק ברדיוס אחד, הדבר היה נראה כאן.

תחנות ללא קואורדינטות אינן יכולות להשתתף ומוצאות מההגדרה המרחבית (קבוצת ההליכה שלהן מוגדרת כקבוצת אופני התחבורה שלהן עצמן); מספרן מודפס.

In [ ]:
# --- Walkable (within-radius) mode sets ----------------------------------
def walkable_mode_sets(frame, radius_m):
    """Union of modes served within `radius_m` of each stop (the stop included).

    Local equirectangular projection; each stop keeps its own radius, so nothing
    chains transitively into over-large 'complexes'.
    """
    geo = frame.dropna(subset=['lat', 'lon'])
    if geo.empty:
        raise ValueError('No stop carries usable coordinates - check nodes.csv (stage 02).')
    lat0 = float(geo['lat'].mean())
    x = geo['lon'].to_numpy(dtype=float) * np.cos(np.deg2rad(lat0)) * 111_320.0
    y = geo['lat'].to_numpy(dtype=float) * 110_540.0
    points = np.column_stack([x, y])
    tree = cKDTree(points)
    neighbours = tree.query_ball_point(points, r=float(radius_m))
    own = [set(s.split('|')) if s else set() for s in geo['modes_served']]
    unions = [set().union(*(own[j] for j in idx)) if idx else set(own[i])
              for i, idx in enumerate(neighbours)]
    return pd.DataFrame({
        'stop_id': geo['stop_id'].to_numpy(),
        'walk_n_modes': [len(u) for u in unions],
        'walk_modes_served': ['|'.join(sorted(u)) for u in unions],
    })


# Sensitivity: how many interchange stops at each candidate radius?
print('{:>8}{:>22}{:>16}'.format('radius', 'interchange stops', 'share of stops'))
sensitivity = {}
for radius in WALK_RADIUS_SENSITIVITY:
    trial = walkable_mode_sets(hubs, radius)
    n_multi = int((trial['walk_n_modes'] >= 2).sum())
    sensitivity[radius] = n_multi
    print('{:>6}m{:>22,}{:>15.2f}%'.format(radius, n_multi, n_multi / len(hubs) * 100))

walk = walkable_mode_sets(hubs, WALK_RADIUS_M)
hubs = hubs.merge(walk, on='stop_id', how='left')
# Stops with no coordinates fall back to their own mode set.
no_geo = hubs['walk_n_modes'].isna()
hubs.loc[no_geo, 'walk_n_modes'] = hubs.loc[no_geo, 'n_modes']
hubs.loc[no_geo, 'walk_modes_served'] = hubs.loc[no_geo, 'modes_served']
hubs['walk_n_modes'] = hubs['walk_n_modes'].astype(int)
hubs['is_interchange'] = hubs['walk_n_modes'] >= 2

print()
print(f'stops with no coordinates (own mode set used): {int(no_geo.sum()):,}')
print(f'interchange stops at {WALK_RADIUS_M} m: {int(hubs["is_interchange"].sum()):,} '
      f'({hubs["is_interchange"].mean() * 100:.2f}% of stops)')
print()
print('walkable mode combinations (top 15):')
print(hubs.loc[hubs['is_interchange'], 'walk_modes_served']
      .value_counts().head(15).to_string())

## 14. טבלת התלות ההדדית המרחבית

אותו מבנה חיתוכים, מחושב מחדש על ההגדרה של מרחק הליכה ונכתב אל `tables/mode_interdependence_walkable.csv` כטבלה נלווית (הקובץ הנדרש `mode_interdependence.csv` שומר על ספירות ה-`stop_id` המשותף הקשיחות, ללא שינוי).

יש לנסח כאן את יחידת המדידה בזהירות. `interchange_stops` עבור הזוג (a, b) הוא **מספר התחנות שסביבתן במרחק הליכה מכילה גם את אופן התחבורה a וגם את b**. זוהי ספירה של נקודות עיגון, ולא של מסופי מעבר פיזיים: תחנת רכבת גדולה המוקפת בשתים-עשרה תחנות אוטובוס תורמת בערך תריסר עוגנים, ולא אחד. הדבר הופך את המספר למדד ל*כמה מרשת האוטובוסים יושבת במרחק הליכה מאופן תחבורה b*, וזוהי הכמות הרלוונטית לחוסן - זו בדיוק קבוצת התחנות שהייתה מאבדת חיבור אילו אופן תחבורה b היה מפסיק לפעול - אך אין זו ספירת תחנות ואין לצטט אותה ככזו.

הניגוד מול הטבלה הקשיחה הוא העיקר: זוגות שאינם חולקים ולו ערך `stop_id` אחד מתגלים כבעלי מאות עוגנים במרחק הליכה.

In [ ]:
# --- tables/mode_interdependence_walkable.csv ----------------------------
walk_sets = [set(s.split('|')) if s else set() for s in hubs['walk_modes_served']]
walk_pair_counts = Counter()
for modes_here in walk_sets:
    for pair in combinations(sorted(modes_here), 2):
        walk_pair_counts[pair] += 1

# observed_modes is ordered by size, so (a, b) is not necessarily alphabetical;
# walk_pair_counts is keyed on sorted pairs, so the lookup key must be sorted too.
interdependence_walk = pd.DataFrame(
    [{'mode_a': a, 'mode_b': b,
      'interchange_stops': walk_pair_counts.get(tuple(sorted((a, b))), 0)}
     for a, b in combinations(observed_modes, 2)]
).sort_values('interchange_stops', ascending=False).reset_index(drop=True)
interdependence_walk.to_csv(TABLES / 'mode_interdependence_walkable.csv', index=False,
                            encoding='utf-8-sig')

comparison = (interdependence
              .merge(interdependence_walk, on=['mode_a', 'mode_b'], how='outer')
              .fillna(0)
              .sort_values('interchange_stops', ascending=False)
              .reset_index(drop=True))
print(f'saved: {TABLES / "mode_interdependence_walkable.csv"}')
print(f'strict shared stop_ids vs walkable anchors within {WALK_RADIUS_M} m:')
print(comparison.to_string(index=False))

## 15. צירוף המדידות המבניות

כדי לשאול האם תחנות רב-אופניות חשובות יותר, דרוש לנו מדד ל"חשיבות" שחושב **ללא כל ידיעה על אופן תחבורה** - אחרת המבחן מעגלי. שני מדדים כאלה כבר קיימים בפרויקט זה, ושניהם חושבו על הגרף הממוזג הכולל את כל אופני התחבורה:

* **Articulation point** (שלב 03): צומת שהסרתו מגדילה את מספר הרכיבים הקשירים. אמירה בינארית וחד-משמעית שהתחנה היא חיתוך ברשת. `articulation_points.csv` מונה אותן; כל תחנה אחרת ב-`nodes.csv` אינה כזו בהגדרה.
* **Betweenness** (שלב 04, עמודה `approx_betweenness`): שיעור המסלולים הקצרים ביותר העוברים דרך התחנה, מוערך מתוך מדגם מקורות, מכיוון ש-betweenness מדויק על גרף בן 30k צמתים אינו בר-ביצוע. הוא רציף, מוטה חזק ימינה, ומלא בתיקו סביב אפס - וזו בדיוק הסיבה שהמבחן שלהלן הוא מבחן דירוג ולא מבחן t.

`degree` מצטרף אף הוא, כמשתנה הריבוד לבדיקת ה-confound בסעיף 17.

החיבור הוא inner join מול `stop_metrics.csv`: תחנה שאינה צומת בגרף הממוזג נטולת betweenness ונטולת סטטוס articulation, ולכן אינה יכולה להשתייך לאף אחת מהקבוצות. מספר התחנות שנשמטו מודפס. מסגרת הניתוח המתקבלת היא זו שכל מבחן שלהלן משתמש בה.

In [ ]:
# --- Join mode facts to the mode-blind structural measurements -----------
structural = metrics[['stop_id', 'degree', BETWEENNESS]].copy()
structural['is_articulation_point'] = structural['stop_id'].isin(articulation_ids)

analysis = hubs.merge(structural, on='stop_id', how='inner')
dropped = len(hubs) - len(analysis)

print(f'stops in the feed                     : {len(hubs):,}')
print(f'stops with structural measurements    : {len(analysis):,}')
print(f'dropped (not vertices of the graph)   : {dropped:,}')
print()
print('group sizes')
print(f'  strict multimodal (shared stop_id)  : {int(analysis["is_multimodal"].sum()):,}')
print(f'  walkable interchange ({WALK_RADIUS_M} m)      : {int(analysis["is_interchange"].sum()):,}')
print(f'  articulation points in the sample   : {int(analysis["is_articulation_point"].sum()):,}'
      f'  ({analysis["is_articulation_point"].mean() * 100:.2f}%)')
print()
print(analysis.groupby('is_multimodal')[[BETWEENNESS, 'degree']]
      .agg(['count', 'median', 'mean']).to_string())

## 16. המבחנים הסטטיסטיים, עם גדלי אפקט

הטענה הנבחנת היא: *תחנות רב-אופניות חשובות מבנית יותר מתחנות חד-אופניות.* היא נבחנת פעמיים - פעם אחת לכל הגדרה של "רב-אופני" - מול שתי תוצאות, ובסך הכול ארבעה מבחנים.

**Betweenness -> Mann-Whitney U.** ה-betweenness כאן רחוק מלהיות נורמלי: הוא משתרע על פני כמה סדרי גודל, רוב התחנות יושבות באפס או בסמוך לו, וגודלי הקבוצות שונים בתכלית. מבחן t היה בלתי ניתן להגנה. Mann-Whitney U משווה בין שתי ההתפלגויות לפי דירוג ושואל האם תחנה רב-אופנית שנבחרה אקראית נוטה להיות מדורגת מעל תחנה חד-אופנית שנבחרה אקראית.

* גודל אפקט: **rank-biserial correlation** $r_{rb} = 2U/(n_1 n_2) - 1$, שהוא Cliff's delta, בתחום $[-1, 1]$; אפס משמעו ששתי הקבוצות משתלבות זו בזו באופן מושלם. לצדו מדווח **common-language effect size** $A = U/(n_1 n_2)$, ההסתברות שתחנה רב-אופנית אקראית עולה על תחנה חד-אופנית אקראית (0.5 = אין הבדל). הפרשנות המקובלת: $|r_{rb}|$ מתחת לכ-0.15 זניח, כ-0.15-0.3 קטן, כ-0.3-0.5 בינוני.

**שיעור ה-articulation point -> מבחן Fisher מדויק.** טבלת 2x2 של קבוצה x צומת חיתוך. מבחן Fisher המדויק הוא המבחן העיקרי מכיוון שהקבוצה הרב-אופנית קטנה ושיעור ה-articulation point הוא אחוזים בודדים, ולכן הספירה הצפויה בתא הקריטי יורדת אל סביבות 5 או מתחתיו - בדיוק המקום שבו קירוב chi-square חדל להיות אמין. סטטיסטיקת ה-chi-square מדווחת אף היא, לשם השוואה מול שאר הספרות, והתא מדפיס את ספירת התא הצפויה המינימלית כדי שהקורא יוכל לשפוט במי להאמין.

* גודל אפקט: ה-**odds ratio** מ-Fisher, בתוספת **phi** $= \sqrt{\chi^2/n}$, שעבור טבלת 2x2 הוא המתאם בין שני המשתנים הבינאריים. phi נמצא בתחום $[0, 1]$ ו*אינו* מנופח על ידי גודל המדגם, מה שחשוב כאן משום ש-$n \approx 30{,}000$ הופך כמעט כל דבר ל"מובהק".
* אי-ודאות בשיעורים עצמם ניתנת כ-**רווחי Wilson ברמת 95%**, שבשונה מקירוב נורמלי מתנהגים בהיגיון עבור ספירות קטנות ושיעורים קרובים לאפס.

**ריבוי השוואות.** ארבעה מבחנים על מערך נתונים אחד, ולכן ערכי ה-p מתוקנים ב-Holm-Bonferroni. Holm ולא Bonferroni גולמי משום שהוא עוצמתי יותר במידה אחידה באותו שיעור שגיאה משפחתי. התיקון אינו משנה דבר מהותי כאן, אך ציונו זול יותר מהגנה על היעדרו.

הכול נכתב אל `tables/multimodal_structural_tests.csv`.

In [ ]:
# --- Test helpers: effect sizes and interval estimates -------------------
def wilson_interval(successes, total, z=1.96):
    """Wilson score interval for a proportion; sane at small n and rates near 0."""
    if total == 0:
        return (float('nan'), float('nan'))
    p = successes / total
    denom = 1 + z**2 / total
    centre = (p + z**2 / (2 * total)) / denom
    half = z * np.sqrt(p * (1 - p) / total + z**2 / (4 * total**2)) / denom
    return (max(0.0, centre - half), min(1.0, centre + half))


def holm_bonferroni(p_values):
    """Holm-Bonferroni family-wise-error adjusted p-values."""
    p = np.asarray(p_values, dtype=float)
    n = p.size
    order = np.argsort(p)
    scaled = p[order] * (n - np.arange(n))
    monotone = np.maximum.accumulate(scaled)
    out = np.empty(n)
    out[order] = np.clip(monotone, 0, 1)
    return out


def effect_size_words(magnitude, scale):
    """Plain-language label for an effect size, so the prose cannot overstate it."""
    cuts = {'rank_biserial': (0.15, 0.30, 0.50), 'phi': (0.10, 0.30, 0.50)}[scale]
    a = abs(magnitude)
    if a < cuts[0]:
        return 'negligible'
    if a < cuts[1]:
        return 'small'
    if a < cuts[2]:
        return 'medium'
    return 'large'


def betweenness_test(frame, flag, definition):
    """Mann-Whitney U on betweenness, with rank-biserial and CLES effect sizes."""
    group = frame[frame[flag]][BETWEENNESS].to_numpy(dtype=float)
    rest = frame[~frame[flag]][BETWEENNESS].to_numpy(dtype=float)
    u_stat, p_value = mannwhitneyu(group, rest, alternative='two-sided')
    pairs = len(group) * len(rest)
    cles = u_stat / pairs
    rank_biserial = 2 * cles - 1
    return {
        'definition': definition,
        'outcome': 'approx_betweenness',
        'test': 'Mann-Whitney U (two-sided)',
        'n_multimodal': len(group),
        'n_single_mode': len(rest),
        'statistic': float(u_stat),
        'p_value': float(p_value),
        'effect_size_name': 'rank_biserial',
        'effect_size': float(rank_biserial),
        'effect_size_reading': effect_size_words(rank_biserial, 'rank_biserial'),
        'cles_prob_multimodal_higher': float(cles),
        'median_multimodal': float(np.median(group)) if len(group) else float('nan'),
        'median_single_mode': float(np.median(rest)) if len(rest) else float('nan'),
        'mean_multimodal': float(np.mean(group)) if len(group) else float('nan'),
        'mean_single_mode': float(np.mean(rest)) if len(rest) else float('nan'),
    }


def articulation_test(frame, flag, definition):
    """Fisher exact (primary) + chi-square on the 2x2 group x cut-vertex table."""
    a = int(((frame[flag]) & (frame['is_articulation_point'])).sum())
    b = int(((frame[flag]) & (~frame['is_articulation_point'])).sum())
    c = int(((~frame[flag]) & (frame['is_articulation_point'])).sum())
    d = int(((~frame[flag]) & (~frame['is_articulation_point'])).sum())
    table = [[a, b], [c, d]]
    odds_ratio, p_fisher = fisher_exact(table)
    chi2, p_chi2, _, expected = chi2_contingency(table)
    n = a + b + c + d
    phi = float(np.sqrt(chi2 / n)) if n else float('nan')
    lo_m, hi_m = wilson_interval(a, a + b)
    lo_s, hi_s = wilson_interval(c, c + d)
    return {
        'definition': definition,
        'outcome': 'is_articulation_point',
        'test': 'Fisher exact (two-sided)',
        'n_multimodal': a + b,
        'n_single_mode': c + d,
        'statistic': float(odds_ratio),
        'p_value': float(p_fisher),
        'effect_size_name': 'phi',
        'effect_size': phi,
        'effect_size_reading': effect_size_words(phi, 'phi'),
        'odds_ratio': float(odds_ratio),
        'chi2': float(chi2),
        'p_value_chi2': float(p_chi2),
        'min_expected_cell': float(expected.min()),
        'rate_multimodal': a / (a + b) if (a + b) else float('nan'),
        'rate_single_mode': c / (c + d) if (c + d) else float('nan'),
        'rate_multimodal_ci': f'[{lo_m:.4f}, {hi_m:.4f}]',
        'rate_single_mode_ci': f'[{lo_s:.4f}, {hi_s:.4f}]',
        'cut_vertices_multimodal': a,
        'cut_vertices_single_mode': c,
    }


DEFINITIONS = [('is_multimodal', 'strict: shares a stop_id'),
               ('is_interchange', f'walkable: modes within {WALK_RADIUS_M} m')]

results = []
for flag, definition in DEFINITIONS:
    results.append(betweenness_test(analysis, flag, definition))
    results.append(articulation_test(analysis, flag, definition))

tests = pd.DataFrame(results)
tests['p_value_holm'] = holm_bonferroni(tests['p_value'])
tests['significant_holm_0.05'] = tests['p_value_holm'] < 0.05
tests.to_csv(TABLES / 'multimodal_structural_tests.csv', index=False, encoding='utf-8-sig')

pd.set_option('display.width', 200)
print(tests[['definition', 'outcome', 'test', 'n_multimodal', 'n_single_mode',
             'p_value', 'p_value_holm', 'effect_size_name', 'effect_size',
             'effect_size_reading']].to_string(index=False))
print()
for row in results:
    print(f'--- {row["definition"]} | {row["outcome"]} ---')
    if row['outcome'] == 'approx_betweenness':
        print(f'    median betweenness  : {row["median_multimodal"]:.3e} (multimodal) '
              f'vs {row["median_single_mode"]:.3e} (single-mode)')
        print(f'    P(multimodal higher): {row["cles_prob_multimodal_higher"]:.3f} '
              f'(0.500 = no difference)')
        print(f'    rank-biserial       : {row["effect_size"]:+.3f} '
              f'-> {row["effect_size_reading"]}')
    else:
        print(f'    cut-vertex rate     : {row["rate_multimodal"] * 100:.2f}% '
              f'{row["rate_multimodal_ci"]} vs {row["rate_single_mode"] * 100:.2f}% '
              f'{row["rate_single_mode_ci"]}')
        print(f'    odds ratio          : {row["odds_ratio"]:.2f}   '
              f'phi = {row["effect_size"]:.4f} -> {row["effect_size_reading"]}')
        print(f'    min expected cell   : {row["min_expected_cell"]:.2f} '
              f'(chi-square is unreliable below ~5; Fisher is the primary test)')
    print(f'    p = {row["p_value"]:.3g}  ->  Holm-adjusted p = '
          f'{float(tests.loc[(tests["definition"] == row["definition"]) & (tests["outcome"] == row["outcome"]), "p_value_holm"].iloc[0]):.3g}')
print()
print('saved:', TABLES / 'multimodal_structural_tests.csv')

## 17. ה-confound המתבקש: האם תחנות מעבר הן פשוט תחנות בעלות degree גבוה?

כל קשר שנמצא לעיל עלול להיות ארטיפקט. תחנות מעבר אינן מדגם אקראי של תחנות - הן יושבות בצמתים, במרכזי ערים, על עורקים ראשיים - וכל אלה מנבאים באופן בלתי תלוי גם degree וגם betweenness. אם לתחנות רב-אופניות פשוט *יש יותר קשתות*, אז מציאתן בשכיחות גבוהה יותר בקרב צמתי חיתוך מלמדת אותנו על degree, ולא על רב-אופניות.

הבדיקה היא ריבוד. התחנות מפוצלות לרצועות degree, טבלת ה-2x2 של קבוצה x צומת חיתוך מחושבת מחדש **בתוך כל רצועה**, והטבלאות הרצועתיות מאוגדות באמצעות ה-odds ratio המשותף של **Cochran-Mantel-Haenszel**

$$OR_{MH} = \frac{\sum_i a_i d_i / n_i}{\sum_i b_i c_i / n_i}$$

המעריך את הקשר בין רב-אופניות ובין היות התחנה צומת חיתוך *בהחזקת degree קבוע*. אילו ה-odds ratio הגולמי היה כולו ארטיפקט של degree, $OR_{MH}$ היה קורס לעבר 1. אם הוא שורד, הקשר אינו degree בלבד.

השיעורים לכל רצועה מודפסים לצד המספר המאוגד, משום שאומדן מאוגד מסתיר הטרוגניות - וכאן הוא אכן מסתיר מעט: האפקט מרוכז ברצועות ה-degree הנמוך, וזה כשלעצמו ההסבר, הנדון בסעיף 21.

In [ ]:
# --- Cochran-Mantel-Haenszel: does the association survive degree control? ---
DEGREE_BANDS = [0, 2, 3, 4, 6, 10, 10_000]


def cmh_analysis(frame, flag, definition, bands=DEGREE_BANDS):
    """Degree-stratified 2x2 tables plus the CMH common odds ratio."""
    work = frame.copy()
    work['degree_band'] = pd.cut(work['degree'], bands)
    rows, num, den = [], 0.0, 0.0
    for band, part in work.groupby('degree_band', observed=True):
        a = int(((part[flag]) & (part['is_articulation_point'])).sum())
        b = int(((part[flag]) & (~part['is_articulation_point'])).sum())
        c = int(((~part[flag]) & (part['is_articulation_point'])).sum())
        d = int(((~part[flag]) & (~part['is_articulation_point'])).sum())
        n = a + b + c + d
        if n == 0:
            continue
        num += a * d / n
        den += b * c / n
        rows.append({'definition': definition, 'degree_band': str(band),
                     'stops': n, 'group_stops': a + b,
                     'rate_multimodal': (a / (a + b)) if (a + b) else float('nan'),
                     'rate_single_mode': (c / (c + d)) if (c + d) else float('nan')})
    common_or = (num / den) if den > 0 else float('nan')
    return pd.DataFrame(rows), float(common_or)


cmh_tables, cmh_summary = [], {}
for flag, definition in DEFINITIONS:
    band_table, common_or = cmh_analysis(analysis, flag, definition)
    cmh_summary[definition] = common_or
    cmh_tables.append(band_table)
    crude = float(tests.loc[(tests['definition'] == definition) &
                            (tests['outcome'] == 'is_articulation_point'),
                            'odds_ratio'].iloc[0])
    print(f'=== {definition} ===')
    print(f'    crude odds ratio                 : {crude:.2f}')
    print(f'    degree-adjusted (CMH) odds ratio : {common_or:.2f}')
    print(band_table.assign(
        rate_multimodal=lambda f: (f['rate_multimodal'] * 100).round(2),
        rate_single_mode=lambda f: (f['rate_single_mode'] * 100).round(2))
        [['degree_band', 'stops', 'group_stops', 'rate_multimodal', 'rate_single_mode']]
        .to_string(index=False))
    print()

cmh_table = pd.concat(cmh_tables, ignore_index=True)

## 18. `tables/transfer_hub_details.csv`

כל מה שנמדד על תחנה ולא נכנס לחוזה בן שמונה העמודות, נכתב פעם אחת, כעת כשגם החיבור המבני וגם קבוצות ההליכה קיימים: נפח שירות משולב, קבוצת אופני התחבורה במרחק הליכה, degree, betweenness ודגל צומת החיתוך. שורה אחת לכל תחנה שיש לה מדידות מבניות, ממוינת כך שהתחנות המעניינות - הכי הרבה אופני תחבורה, ולאחר מכן הכי הרבה שירות - נמצאות בראש.

זו הטבלה שיש לפתוח כאשר סיווגה של תחנה מסוימת נראה מפתיע: היא מציגה, בשורה אחת, כמה אופני תחבורה משרתים אותה במובן הקשיח, כמה במרחק הליכה, כמה עמוסה היא, והאם הרשת הממוזגת נשברת בעת הסרתה.

In [ ]:
# --- tables/transfer_hub_details.csv -------------------------------------
details = analysis[['stop_id', 'stop_name', 'lat', 'lon', 'region',
                    'n_modes', 'modes_served', 'is_multimodal',
                    'walk_n_modes', 'walk_modes_served', 'is_interchange',
                    'total_stop_calls', 'degree', BETWEENNESS,
                    'is_articulation_point']].copy()
details = details.sort_values(['n_modes', 'walk_n_modes', 'total_stop_calls'],
                              ascending=False).reset_index(drop=True)
details.to_csv(TABLES / 'transfer_hub_details.csv', index=False, encoding='utf-8-sig')

print(f'{len(details):,} rows -> {TABLES / "transfer_hub_details.csv"}')
print()
print(f'--- top {TOP_N_HUBS} walkable interchange stops by service volume ---')
print(details[details['is_interchange']]
      .sort_values('total_stop_calls', ascending=False)
      .head(TOP_N_HUBS)[['stop_id', 'stop_name', 'region', 'walk_n_modes',
                         'walk_modes_served', 'total_stop_calls',
                         'is_articulation_point']]
      .to_string(index=False))

## 19. איור 1 - מצאי המוקדים

שלושה פאנלים, משמאל לימין:

1. **כמה תחנות משרתות כמה אופני תחבורה**, תחת שתי ההגדרות, על ציר y לוגריתמי. הסקאלה הלוגריתמית אינה קוסמטית: עמודת התחנות החד-אופניות גבוהה בשניים עד שלושה סדרי גודל מכל השאר, ועל ציר לינארי העמודות הרב-אופניות היו בלתי נראות. שני הצבעים הם כל טיעונו של סעיף 13 בתמונה אחת.
2. **המוקדים המובילים לפי נפח שירות משולב**, עמודות אופקיות עם שמות בעברית. אופקיות משום שהשמות ארוכים; ממוינות כך שהעמוס ביותר בראש.
3. **היכן נמצאות תחנות המעבר**, לפי מחוז. מדווח כ*שיעור* מתוך תחנות כל מחוז ולא כספירה, משום שלמחוז המרכז יש סדר גודל יותר תחנות מכל מחוז אחר, והספירות הגולמיות לא היו אומרות דבר מלבד "המרכז גדול".

In [ ]:
# --- Figure 1: hub inventory ---------------------------------------------
fig, axes = plt.subplots(1, 3, figsize=(17, 5.2))

# Panel 1: distribution of modes per stop, both definitions
ax = axes[0]
strict_counts = hubs['n_modes'].value_counts().sort_index()
walk_counts = hubs['walk_n_modes'].value_counts().sort_index()
levels = sorted(set(strict_counts.index) | set(walk_counts.index))
x = np.arange(len(levels))
width = 0.38
bars1 = ax.bar(x - width / 2, [strict_counts.get(k, 0) for k in levels], width,
               label='shares a stop_id', color='#3b6ea5')
bars2 = ax.bar(x + width / 2, [walk_counts.get(k, 0) for k in levels], width,
               label=f'within {WALK_RADIUS_M} m', color='#e07b39')
for bars in (bars1, bars2):
    ax.bar_label(bars, labels=[f'{int(b.get_height()):,}' if b.get_height() else ''
                               for b in bars], fontsize=7, padding=2, rotation=90)
ax.set_yscale('log')
ax.set_ylim(0.6, max(strict_counts.max(), walk_counts.max()) * 20)
ax.set_xticks(x)
ax.set_xticklabels(levels)
ax.set_xlabel('modes serving the stop')
ax.set_ylabel('stops (log scale)')
ax.set_title('How many modes serve a stop?')
ax.legend(fontsize=8)

# Panel 2: top hubs by combined service volume
ax = axes[1]
top = (details[details['is_interchange']]
       .sort_values('total_stop_calls', ascending=False)
       .head(TOP_N_HUBS)
       .iloc[::-1])
ax.barh(range(len(top)), top['total_stop_calls'], color='#5a9e6f')
ax.set_yticks(range(len(top)))
ax.set_yticklabels([f'{name} ({n})' for name, n in
                    zip(top['stop_name'], top['walk_n_modes'])], fontsize=8)
ax.set_xlabel('combined scheduled stop calls')
ax.set_title(f'Busiest interchange stops (modes within {WALK_RADIUS_M} m)')

# Panel 3: interchange share by region
ax = axes[2]
by_region = (hubs.assign(region=hubs['region'].replace('', 'unknown'))
             .groupby('region')
             .agg(stops=('stop_id', 'size'),
                  interchange=('is_interchange', 'sum'),
                  multimodal=('is_multimodal', 'sum')))
by_region['interchange_share'] = by_region['interchange'] / by_region['stops'] * 100
by_region = by_region.sort_values('interchange_share', ascending=False)
bars = ax.bar(range(len(by_region)), by_region['interchange_share'], color='#8d6cab')
ax.bar_label(bars, labels=[f'{v:.1f}%' for v in by_region['interchange_share']],
             fontsize=8, padding=2)
ax.set_xticks(range(len(by_region)))
ax.set_xticklabels([f'{idx}\n(n={int(row.stops):,})'
                    for idx, row in by_region.iterrows()], fontsize=8)
ax.set_ylabel("% of the region's stops")
ax.set_title('Interchange stops by region')

fig.suptitle('Multimodal transfer hubs in the Israeli GTFS feed', fontsize=13)
fig.tight_layout(rect=(0, 0, 1, 0.94))
fig.savefig(FIGURES / 'hub_inventory.png', dpi=FIG_DPI)
plt.show()
print('saved:', FIGURES / 'hub_inventory.png')

## 20. איור 2 - גרף התלות ההדדית בין אופני התחבורה

שני שרטוטים של אותו אובייקט תחת שתי ההגדרות. הצמתים הם אופני תחבורה, בגודל לפי מספר התחנות שאותו אופן תחבורה משרת (בסקאלה לוגריתמית, אחרת אוטובוס היה הצומת הנראה היחיד); קשתות משורטטות רק היכן שהמשקל שונה מאפס, ברוחב פרופורציונלי לשורש הריבועי של המשקל, והמשקל מודפס על הקשת. פריסה מעגלית קבועה שומרת על השוואתיות בין שני הפאנלים ועל שחזוריות האיור - פריסת spring הייתה מזיזה את הצמתים בין הרצות ובין פאנלים ללא כל תועלת אנליטית.

הפאנל השמאלי הוא הגרף הקשיח: קשת יחידה. הפאנל הימני הוא גרף מרחק ההליכה, והוא כוכב קשיר שמרכזו אוטובוס. צורה זו היא כשלעצמה אמירת החוסן של מחברת זו: **יכולת המעבר של כל אופן תחבורה אחר עוברת דרך רשת האוטובוסים**, ואף אחד מאופני התחבורה הקטנים אינו מתחבר לאחרים אלא דרכה.

In [ ]:
# --- Figure 2: mode interdependence, both definitions --------------------
def draw_mode_graph(ax, pair_weights, node_sizes, title, colour):
    G = nx.Graph()
    for m in observed_modes:
        G.add_node(m, stops=node_sizes.get(m, 0))
    for (a, b), w in pair_weights.items():
        if w > 0:
            G.add_edge(a, b, weight=w)
    pos = nx.circular_layout(sorted(G.nodes()))
    sizes = [300 + 1400 * np.log10(1 + G.nodes[n]['stops']) / 5 for n in G.nodes()]
    nx.draw_networkx_nodes(G, pos, ax=ax, node_size=sizes, node_color=colour,
                           edgecolors='white', linewidths=1.5)
    if G.number_of_edges():
        weights = [G[u][v]['weight'] for u, v in G.edges()]
        widths = [1.0 + 6.0 * np.sqrt(w) / np.sqrt(max(weights)) for w in weights]
        nx.draw_networkx_edges(G, pos, ax=ax, width=widths, alpha=0.55,
                               edge_color='#444444')
        nx.draw_networkx_edge_labels(
            G, pos, ax=ax, font_size=8,
            edge_labels={(u, v): f'{G[u][v]["weight"]:,}' for u, v in G.edges()})
    nx.draw_networkx_labels(G, pos, ax=ax, font_size=8,
                            labels={n: f'{n}\n({G.nodes[n]["stops"]:,} stops)'
                                    for n in G.nodes()})
    ax.set_title(title, fontsize=11)
    ax.axis('off')
    return G


strict_weights = {(r.mode_a, r.mode_b): int(r.shared_stops)
                  for r in interdependence.itertuples()}
walk_weights = {(r.mode_a, r.mode_b): int(r.interchange_stops)
                for r in interdependence_walk.itertuples()}
strict_sizes = {m: len(s) for m, s in mode_stop_sets.items()}

n_strict_edges = sum(1 for w in strict_weights.values() if w > 0)
n_walk_edges = sum(1 for w in walk_weights.values() if w > 0)

fig, axes = plt.subplots(1, 2, figsize=(15, 7.2))
G_strict = draw_mode_graph(axes[0], strict_weights, strict_sizes,
                           'Strict: modes sharing the same stop_id\n'
                           f'({n_strict_edges} edge(s))',
                           '#3b6ea5')
G_walk = draw_mode_graph(axes[1], walk_weights, strict_sizes,
                         f'Walkable: modes within {WALK_RADIUS_M} m of each other\n'
                         f'({n_walk_edges} edges)',
                         '#e07b39')
fig.suptitle('Mode-interdependence graph: nodes are transport modes, '
             'edge weight = shared stops', fontsize=13)
fig.tight_layout(rect=(0, 0, 1, 0.93))
fig.savefig(FIGURES / 'mode_interdependence_graph.png', dpi=FIG_DPI)
plt.show()

print('strict graph : {} nodes, {} edges, connected = {}'.format(
    G_strict.number_of_nodes(), G_strict.number_of_edges(),
    nx.is_connected(G_strict) if G_strict.number_of_nodes() else False))
print('walkable graph: {} nodes, {} edges, connected = {}'.format(
    G_walk.number_of_nodes(), G_walk.number_of_edges(),
    nx.is_connected(G_walk) if G_walk.number_of_nodes() else False))
print('saved:', FIGURES / 'mode_interdependence_graph.png')

## 21. איור 3 - היכן נמצאים המוקדים

פיזור גאוגרפי ובו כל התחנות באפור בהיר כרקע, שתי הגדרות המוקד מונחות מעליו, והמוקדים הגדולים ביותר מתויגים. יחס הממדים מתוקן באמצעות `cos(latitude)` כדי שהמדינה לא תימתח לרוחב.

* **פאנל שמאלי** - תחנות רב-אופניות במובן הקשיח. הן מפוזרות על פני המדינה ללא כל היגיון של מעבר, וזה בדיוק מה שמצפים לו אם ה"רב-אופניות" היא ארטיפקט רישום של שירותים מבוססי-ביקוש החולקים תחנות אוטובוס.
* **פאנל ימני** - תחנות מעבר במרחק הליכה, צבועות לפי מספר אופני התחבורה הנגישים. אלה מתאשכלות בעוצמה אל עבר עורקי תל אביב, ירושלים וחיפה ולאורך ציר הרכבת, וזה נראה כמו גאוגרפיית מעבר אמיתית. `MAP_ANNOTATE` העמוסות ביותר מתויגות בשם.

השוואת שני הפאנלים היא הדרך המהירה ביותר להבין מדוע המחברת אינה עוצרת בהגדרה הקשיחה.

In [ ]:
# --- Figure 3: hub geography ---------------------------------------------
geo = hubs.dropna(subset=['lat', 'lon'])
aspect = 1 / np.cos(np.deg2rad(float(geo['lat'].mean())))

fig, axes = plt.subplots(1, 2, figsize=(14, 10))

# Left: strict multimodal stops
ax = axes[0]
ax.scatter(geo['lon'], geo['lat'], s=1, alpha=0.12, linewidths=0, color='#b0b0b0',
           label=f'all stops ({len(geo):,})')
strict_geo = geo[geo['is_multimodal']]
ax.scatter(strict_geo['lon'], strict_geo['lat'], s=28, alpha=0.9, linewidths=0,
           color='#c0504d', label=f'multimodal stop_id ({len(strict_geo):,})')
ax.set_aspect(aspect)
ax.set_title('Strict: stops served by 2+ modes\nunder the same stop_id', fontsize=11)
ax.set_xlabel('Longitude')
ax.set_ylabel('Latitude')
ax.legend(loc='lower left', fontsize=8, markerscale=2.5, framealpha=0.9)

# Right: walkable interchange stops, coloured by walkable mode count
ax = axes[1]
ax.scatter(geo['lon'], geo['lat'], s=1, alpha=0.12, linewidths=0, color='#b0b0b0')
inter_geo = geo[geo['is_interchange']].sort_values('walk_n_modes')
scatter = ax.scatter(inter_geo['lon'], inter_geo['lat'], s=18, alpha=0.85, linewidths=0,
                     c=inter_geo['walk_n_modes'], cmap='viridis')
cbar = fig.colorbar(scatter, ax=ax, fraction=0.035, pad=0.02)
cbar.set_label('modes within {} m'.format(WALK_RADIUS_M))

label_me = (details[details['is_interchange']]
            .dropna(subset=['lat', 'lon'])
            .sort_values('total_stop_calls', ascending=False)
            .head(MAP_ANNOTATE))
for _, row in label_me.iterrows():
    ax.annotate(row['stop_name'], (row['lon'], row['lat']),
                textcoords='offset points', xytext=(6, 4), fontsize=7,
                bbox=dict(boxstyle='round,pad=0.2', fc='white', ec='none', alpha=0.75))
ax.set_aspect(aspect)
ax.set_title(f'Walkable: modes reachable within {WALK_RADIUS_M} m\n'
             f'({len(inter_geo):,} interchange stops, top {MAP_ANNOTATE} labelled)',
             fontsize=11)
ax.set_xlabel('Longitude')
ax.set_ylabel('Latitude')

fig.suptitle('Where the transport modes actually meet', fontsize=13)
fig.tight_layout(rect=(0, 0, 1, 0.95))
fig.savefig(FIGURES / 'transfer_hub_map.png', dpi=FIG_DPI)
plt.show()
print('saved:', FIGURES / 'transfer_hub_map.png')

## 22. איור 4 - האם תחנות רב-אופניות חשובות מבנית יותר?

המבחנים של סעיף 16, בשרטוט.

* **שמאל** - betweenness לפי קבוצה, כתרשימי קופסה על ציר לוגריתמי עם `log10(betweenness + eps)`. ההיסט נחוץ משום שלחלק גדול מהתחנות יש betweenness אפס בדיוק ו-`log(0)` אינו מוגדר; ההזזה מוחלת באופן זהה על שתי הקבוצות ולכן אינה יכולה לשנות את הסדר, והמבחן עצמו הורץ על הערכים הגולמיים ולא על אלה שהותמרו. הקופסאות מחורצות (notched), כך שחריצים שאינם חופפים מעידים בקירוב על הבדל מובהק בחציונים.
* **ימין** - שיעור ה-articulation point לפי קבוצה עם **מוטות שגיאה של Wilson ברמת 95%**, וכאן גודלה הקטן של הקבוצה הרב-אופנית הקשיחה נעשה גלוי בדמות רווח רחב. ההערה על כל זוג נותנת את ה-odds ratio ואת ערך ה-p המתוקן ב-Holm, כך שהאיור נושא את אותם מספרים כמו הטבלה ואינו יכול לסטות ממנה.

שני הפאנלים יחד הם התשובה לשאלת המחקר של המחברת, והם אינם עונים עליה באותו אופן עבור שתי ההגדרות.

In [ ]:
# --- Figure 4: structural importance, by definition ----------------------
EPS = 1e-9
fig, axes = plt.subplots(1, 2, figsize=(14, 5.6))

# Left: betweenness distributions
ax = axes[0]
box_data, box_labels = [], []
for flag, definition in DEFINITIONS:
    for is_group, tag in [(True, 'multimodal'), (False, 'single-mode')]:
        values = analysis.loc[analysis[flag] == is_group, BETWEENNESS].to_numpy(dtype=float)
        box_data.append(np.log10(values + EPS))
        short = 'strict' if flag == 'is_multimodal' else f'{WALK_RADIUS_M} m'
        box_labels.append(f'{short}\n{tag}\n(n={len(values):,})')
bp = ax.boxplot(box_data, showfliers=False, notch=True, patch_artist=True)
ax.set_xticks(range(1, len(box_labels) + 1))
ax.set_xticklabels(box_labels)
for patch, colour in zip(bp['boxes'], ['#c0504d', '#9dbbd6', '#e07b39', '#9dbbd6']):
    patch.set_facecolor(colour)
ax.set_ylabel('log10(approx_betweenness + 1e-9)')
ax.set_title('Betweenness by group (Mann-Whitney U)')
ax.tick_params(axis='x', labelsize=8)

# Right: articulation-point rates with Wilson intervals
ax = axes[1]
positions, heights, errors, labels, colours = [], [], [], [], []
for k, (flag, definition) in enumerate(DEFINITIONS):
    row = tests[(tests['definition'] == definition) &
                (tests['outcome'] == 'is_articulation_point')].iloc[0]
    for j, (rate, n_group, tag, colour) in enumerate([
            (row['rate_multimodal'], row['n_multimodal'], 'multimodal',
             '#c0504d' if k == 0 else '#e07b39'),
            (row['rate_single_mode'], row['n_single_mode'], 'single-mode', '#9dbbd6')]):
        successes = int(round(rate * n_group))
        lo, hi = wilson_interval(successes, int(n_group))
        positions.append(k * 2.6 + j)
        heights.append(rate * 100)
        errors.append([(rate - lo) * 100, (hi - rate) * 100])
        short = 'strict' if flag == 'is_multimodal' else f'{WALK_RADIUS_M} m'
        labels.append(f'{short}\n{tag}')
        colours.append(colour)
err = np.array(errors).T
ax.bar(positions, heights, color=colours, width=0.8,
       yerr=err, capsize=4, error_kw={'ecolor': '#333333', 'lw': 1.2})
ax.set_xticks(positions)
ax.set_xticklabels(labels, fontsize=8)
ax.set_ylabel('% of stops that are articulation points')
ax.set_title('Cut-vertex rate by group (Fisher exact, Wilson 95% CI)')
top_y = max(h + e for h, e in zip(heights, err[1])) * 1.25
ax.set_ylim(0, top_y)
for k, (flag, definition) in enumerate(DEFINITIONS):
    row = tests[(tests['definition'] == definition) &
                (tests['outcome'] == 'is_articulation_point')].iloc[0]
    ax.text(k * 2.6 + 0.5, top_y * 0.92,
            f'OR = {row["odds_ratio"]:.2f}\nHolm p = {row["p_value_holm"]:.2g}\n'
            f'phi = {row["effect_size"]:.3f} ({row["effect_size_reading"]})',
            ha='center', va='top', fontsize=8,
            bbox=dict(boxstyle='round,pad=0.3', fc='white', ec='#999999', alpha=0.9))

fig.suptitle('Are multimodal stops structurally more important than single-mode stops?',
             fontsize=13)
fig.tight_layout(rect=(0, 0, 1, 0.93))
fig.savefig(FIGURES / 'structural_importance.png', dpi=FIG_DPI)
plt.show()
print('saved:', FIGURES / 'structural_importance.png')

## 23. `multimodal_summary.json`

הפלט הנדרש השלישי: כל מספר מפתח בקובץ אחד קריא למכונה, כך שהדוח לעולם אינו צריך לגזור מחדש נתון ביד מתוך CSV. הוא נושא את מצאי אופני התחבורה, את שני מבני התלות ההדדית, את ספירות המוקדים תחת שתי ההגדרות, את תוצאות המבחנים המלאות עם גדלי אפקט, את ה-odds ratio המתוקנים ל-degree, את בדיקת הרגישות לרדיוס, ואת המוקדים המובילים לפי נפח שירות. `default=float` מטפל בסקלרים של numpy; `ensure_ascii=False` שומר על קריאות שמות התחנות בעברית בתוך הקובץ עצמו.

In [ ]:
# --- multimodal_summary.json ---------------------------------------------
top_records = (details.sort_values(['walk_n_modes', 'total_stop_calls'], ascending=False)
               .head(TOP_N_HUBS)[['stop_id', 'stop_name', 'region', 'n_modes',
                                  'modes_served', 'walk_n_modes', 'walk_modes_served',
                                  'total_stop_calls', 'is_articulation_point']]
               .to_dict(orient='records'))

summary = {
    'stage': '17_multimodal_transfer_hubs',
    'walk_radius_m': WALK_RADIUS_M,
    'stops_in_feed': int(len(hubs)),
    'stops_with_structural_metrics': int(len(analysis)),
    'stop_time_rows_read': int(stream_stats['rows_read']),
    'modes': {
        m: {'route_type': next((r['route_type'] for _, r in mode_meta.iterrows()
                                if r['mode_label'] == m), ''),
            'stops': int(len(mode_stop_sets.get(m, ()))),
            'stop_calls': int(sum(stop_calls.get(m, {}).values())),
            'trips_observed': int(stream_stats['trips_observed'].get(m, 0))}
        for m in observed_modes},
    'strict_definition': {
        'multimodal_stops': int(hubs['is_multimodal'].sum()),
        'share_of_stops': float(hubs['is_multimodal'].mean()),
        'max_modes_at_one_stop': int(hubs['n_modes'].max()),
        'mode_combinations': hubs.loc[hubs['is_multimodal'], 'modes_served']
                                 .value_counts().to_dict(),
        'mode_pairs_sharing_stops': int((interdependence['shared_stops'] > 0).sum()),
        'interdependence': interdependence.to_dict(orient='records'),
    },
    'walkable_definition': {
        'interchange_stops': int(hubs['is_interchange'].sum()),
        'share_of_stops': float(hubs['is_interchange'].mean()),
        'max_modes_within_radius': int(hubs['walk_n_modes'].max()),
        'radius_sensitivity': {f'{r}m': int(v) for r, v in sensitivity.items()},
        'mode_pairs_within_radius': int((interdependence_walk['interchange_stops'] > 0).sum()),
        'interdependence': interdependence_walk.to_dict(orient='records'),
    },
    'structural_tests': tests.to_dict(orient='records'),
    'degree_adjusted_odds_ratios': cmh_summary,
    'top_hubs': top_records,
    'caveats': [
        'GTFS stop_ids are per-mode records: a rail platform and the bus bay outside '
        'it are different stops, so the strict shared-stop_id definition understates '
        'real interchange.',
        'The walkable definition depends on an arbitrary radius; 100/150/250 m are '
        'reported in radius_sensitivity.',
        'Betweenness is stage 04 approx_betweenness (sampled sources), not exact.',
        'Articulation points and betweenness come from the merged all-mode graph, '
        'which is numerically dominated by bus.',
        'Effect sizes, not p-values, carry the conclusion: n ~ 30,000 makes almost '
        'any difference statistically significant.',
    ],
}
with open(STAGE / 'multimodal_summary.json', 'w', encoding='utf-8') as handle:
    json.dump(summary, handle, ensure_ascii=False, indent=2, default=float)

print('saved:', STAGE / 'multimodal_summary.json')
print(json.dumps({k: summary[k] for k in ('stops_in_feed', 'walk_radius_m')},
                 ensure_ascii=False))
print('strict multimodal stops   :', summary['strict_definition']['multimodal_stops'])
print('walkable interchange stops:', summary['walkable_definition']['interchange_stops'])

## 24. מדוע תחנות מעבר חשובות לחוסן

המנגנון פשוט, והוא הסיבה לקיומה של מחברת זו.

**תחנה חד-אופנית כושלת ברשת אחת. תחנת מעבר כושלת בכמה רשתות בבת אחת.** הסירו תחנת אוטובוס רגילה וגרף האוטובוסים מאבד צומת; כל אופן תחבורה אחר אינו נפגע, ונוסעים שלא השתמשו באותה תחנה אינם מבחינים בדבר. הסירו תחנה שבה אוטובוס פוגש רכבת ושני דברים קורים בו-זמנית: רשת האוטובוסים מאבדת צומת, *וגם* רשת הרכבת מאבדת את חיבור ההגעה ברגל היחיד שלה באותה נקודה. קו הרכבת עדיין פועל - קשתותיו שלו שלמות - אך תחנת רכבת שלא ניתן להגיע אליה באוטובוס כשלה, מנקודת מבטו של הנוסע, באופן חלקי. כשל מדורג ברשתות מצומדות אינו מטפורה כאן; זו משמעותו של "תחנת האוטובוס המזינה סגורה" עבור מי שעומד על הרציף.

**הצימוד א-סימטרי, והא-סימטריה נראית באיור 2.** גרף התלות ההדדית של מרחק ההליכה הוא **כוכב שמרכזו אוטובוס**. רכבת, רכבת קלה, רכבל וקווי מוניות השירות מתחברים כל אחד לאוטובוס, ולמעשה לא זה לזה. כך שרשת האוטובוסים אינה רק השכבה הגדולה ביותר, היא *מצע המעבר*: היא זו שדרכה מתחבר כל אופן תחבורה אחר. מחברת 14 הראתה שאוטובוס הוא אופן התחבורה היחיד עם מסלולים חלופיים אמיתיים; מחברת זו מראה שהוא גם היחיד המחבר את כל האחרים. שיבוש בשירות האוטובוסים בעורק כלשהו פוגע אפוא בנגישות של כל אופן תחבורה אחר באותו עורק בעת ובעונה אחת, בעוד שכשל ברכבת מותיר את רשת האוטובוסים שלמה מבנית.

**תחנות מעבר הן המקום שבו הרדנדנטיות מתחלקת באופן לא אחיד.** מחברת 14 מצאה שאופני התחבורה בעלי המסילה הקבועה הם כמעט גרפי-מסלול: כמעט כל תחנת ביניים היא צומת חיתוך של הרשת שלה עצמה. תחנות אלה הן גם, מעצם הבנייה, אלה המופיעות בקבוצת תחנות המעבר במרחק הליכה. כך שהתחנות שבהן אופני התחבורה נפגשים יורשות את תכונות הרדנדנטיות ה*גרועות* ביותר של אופן התחבורה הדליל ביותר הנוכח, ולא את התכונות הטובות ביותר של הצפוף ביותר. טבלת רצועות ה-degree בסעיף 17 מראה זאת ישירות: עודף שיעור צומתי החיתוך בקרב תחנות המעבר מרוכז ברצועות ה-**degree הנמוך** - תחנות בעלות שניים או שלושה שכנים, שאף על פי כן מעגנות מעבר בין אופני תחבורה. אלה בדיוק התחנות ה"שקטות אך קריטיות" שרשימת עדיפויות מבוססת-תנועה מפספסת, אותה תבנית שמחברת 12 מצאה עבור הרכבת.

**אבל - וכאן יש להציג את הראיות ביושר - גדלי האפקט קטנים.** תחת ההגדרה הקשיחה אין למעשה *שום* הבדל בשיעור צמתי החיתוך: ה-odds ratio קרוב ל-1, ערך ה-p של Fisher רחוק ממובהקות, phi הוא בסדר גודל של 0.001, וה-odds ratio המתוקן ל-degree הוא אם כבר מתחת ל-1. ההבדל ב-betweenness תחת אותה הגדרה מובהק סטטיסטית, אך ה-rank-biserial correlation הוא סביב 0.25 - אפקט קטן, וכזה שמצטמצם עוד יותר ברגע שלוקחים degree בחשבון. התשובה הקשיחה לשאלה "האם תחנות רב-אופניות חשובות יותר?" היא: **בקושי, ובעיקר משום שהן נמצאות במקומות עמוסים יותר.**

תחת ההגדרה של מרחק הליכה, הקשר לסטטוס צומת חיתוך חזק בהרבה - שיעור צמתי חיתוך גבוה פי כמה מקו הבסיס, odds ratio הרבה מעל 1, והוא שורד ריבוד לפי degree עם odds ratio של CMH רחוק מ-1. אך גם שם phi הוא סביב 0.16, שלפי המוסכמה המקובלת הוא אפקט *קטן*, וכיוון ההשוואה של ה-betweenness **מתהפך**: לתחנות מעבר במרחק הליכה יש betweenness חציוני *נמוך יותר* מאשר לתחנות רגילות. אין זו סתירה ואין להחליק על כך. משמעות הדבר היא שתחנות אלה הן חיתוכים ב*פריפריה דלילה* - תחנות רכבת בשולי אזורים בנויים, תחנות רכבת קלה על עורק לינארי - ולא מתווכים במרקם המטרופוליני הצפוף שבו נצבר betweenness. הן קריטיות במובן שהסרתן מנתקת משהו; הן אינן קריטיות במובן שתנועה רבה זורמת דרכן.

**מה שכל זה מרמז לגבי תעדוף.** מפעיל המדרג תחנות לצורך חיזוק אינו צריך להשתמש ב"משרתת יותר מאופן תחבורה אחד" כקריטריון בפני עצמו - על פי הראיות הללו הוא קונה מעט מאוד, ותחת ההגדרה הקשיחה של GTFS הוא קונה כמעט כלום. מה שכן יש להשתמש בו הוא ה*צירוף* שנמצא כאן: **תחנה שהיא צומת חיתוך, יושבת במרחק הליכה מאופן תחבורה שני, ונושאת נפח מתוזמן משמעותי.** `tables/transfer_hub_details.csv` נושא בדיוק את שלוש העמודות הללו לכל תחנה, כך שניתן לברור את הצירוף ישירות. קבוצה זו קטנה, וחבריה אינם נראים מרשימים בדירוג מבוסס-תנועה - וזו כל הסיבה שכדאי להריץ ניתוח מבני.

**מגבלות המסייגות את כל האמור לעיל.**

* **ההגדרה הקשיחה מודדת את אופן הרישום ב-GTFS, ולא תחבורה.** זוהי המגבלה הדומיננטית. זו הסיבה שהמחברת מדווחת את שתי ההגדרות ומסרבת לבחור באחת מהן כ"התשובה".
* **רדיוס ההליכה הוא בחירת מידול.** 150 מ' ניתן להגנה; 100 מ' ו-250 מ' נותנים גדלי קבוצות שונים (סעיף 13 מדפיס את שלושתם). *כיוון* כל תוצאה יציב על פני שלושת הרדיוסים, אך הסדרי גודל אינם.
* **מרחק ההליכה הוא מרחק אווירי.** לא ממודל שום מחסום - כביש מהיר, חתך מסילה או נהר בין שתי תחנות המרוחקות 120 מ' זו מזו הופכים אותן, הלכה למעשה, ללא-מסוף-מעבר.
* **Articulation points ו-betweenness מגיעים מהגרף הממוזג**, הנשלט מספרית על ידי האוטובוס. "צומת חיתוך" כאן הוא חיתוך של רשת סמיכות הנסיעות המשולבת, ולא של אופן תחבורה בודד כלשהו.
* **ה-betweenness מקורב** (מקורות מדגמיים, שלב 04). בדיקת היציבות של שלב 04 עצמה חוסמת את שגיאת הדגימה; היא קטנה יחסית להבדלים בין הקבוצות המדווחים כאן, אך אינה אפס.
* **גדלי האפקט נושאים את המסקנה, לא ערכי ה-p.** עם n ~ 30,000, הבדל חסר כל משמעות מעשית מגיע ל-p < 1e-50 באופן שגרתי. לכן כל מבחן לעיל מדווח עם גודל אפקט ועם קריאה שלו בשפה פשוטה, והטקסט אומר "קטן" היכן שהוא קטן.

## מסקנות

* **אופני התחבורה בישראל כמעט אף פעם אינם חולקים תחנת GTFS.** רק שבריר זעיר מהתחנות משרת יותר מאופן תחבורה אחד, אף תחנה אינה משרתת יותר משניים, והזוג המעורב הוא כמעט תמיד אוטובוס + אוטובוס מבוסס-ביקוש - שני קודים לאותה תחנת אוטובוס פיזית. **אוטובוס ורכבת חולקים אפס ערכי `stop_id`.** כל ניתוח המגדיר "מוקד מעבר" כ"מזהה תחנה משותף" בקובץ זה מודד מוסכמת רישום נתונים.
* **מעבר להגדרה של מרחק הליכה חושף את גאוגרפיית המעבר.** בתוך 150 מ', מאות תחנות מגיעות לאופן תחבורה שני, זוגות אופני התחבורה אוטובוס-רכבת, אוטובוס-רכבת קלה, אוטובוס-רכבל ואוטובוס-מונית הופכים כולם ללא-ריקים, והמוקדים מתאשכלים בדיוק היכן שהיה מצופה: בגלעיני תל אביב, ירושלים וחיפה ולאורך ציר הרכבת.
* **גרף התלות ההדדית הוא כוכב שמרכזו אוטובוס.** כל אופן תחבורה משני מתחבר לאוטובוס ולמעשה לא ליותר מכך. אוטובוס אינו רק השכבה הגדולה ביותר של הרשת הישראלית, הוא המצע שדרכו נגיש כל אופן תחבורה אחר - מה שהופך שיבוש באוטובוסים לסוג השיבוש היחיד הפוגע בכל אופני התחבורה בבת אחת.
* **הטענה בדבר החשיבות המבנית נתמכת רק במחצית, והסיכום הכן הוא "קטן".** תחנות רב-אופניות במובן הקשיח *אינן* בעלות סבירות גבוהה יותר להיות צמתי חיתוך (odds ratio קרוב ל-1, ערך p של Fisher רחוק ממובהקות, phi ~ 0.001, odds ratio מתוקן ל-degree מתחת ל-1). ה-betweenness שלהן גבוה יותר באופן מובהק לפי מבחן דירוג, אך עם rank-biserial correlation סביב 0.25 - אפקט קטן, המוסבר במידה רבה בכך שתחנות אלה יושבות במקומות צפופים יותר.
* **לתחנות מעבר במרחק הליכה *כן* יש סבירות גבוהה בהרבה להיות צמתי חיתוך**, בפקטור של כמה מונים, והקשר שורד ריבוד לפי degree. אך phi הוא סביב 0.16 - עדיין אפקט קטן בסקאלה המקובלת - וה-betweenness החציוני שלהן *נמוך* מזה של תחנות רגילות. הן חיתוכים בפריפריה הדלילה, ולא מתווכות בגלעין הצפוף. שתי העובדות מדווחות משום שכל אחת מהן לבדה הייתה מטעה.
* **המסקנה התפעולית היא צירוף, ולא דגל בודד.** "משרתת שני אופני תחבורה" הוא קריטריון חלש לחיזוק. "היא צומת חיתוך, יש לה אופן תחבורה שני במרחק הליכה, והיא נושאת נפח מתוזמן אמיתי" הוא קריטריון חזק, והוא בורר קבוצת תחנות קטנה, ספציפית ולא מובנת מאליה - שאותה `tables/transfer_hub_details.csv` בנוי לאפשר לקורא לחלץ בסינון אחד.
* **כל האמור כאן הוא צד ההיצע.** ל-GTFS אין נתוני ביקוש, ולכן "נפח שירות" הוא עצירות מתוזמנות ו"חשוב" משמעו חשוב בגרף לוח הזמנים. תחנה עם ספירת עצירות נמוכה שבמקרה היא מסוף המעבר היחיד של עיירה שלמה זוכה למשקל חסר בכל מדד במחברת זו, ושום כמות של סטטיסטיקה אינה מתקנת זאת - רק נתוני ביקוש היו מתקנים.